# Graph-Enhanced CodeT5+ Embeddings for Software Fault Severity Prediction

**Reproducibility notebook.** One notebook for the whole experiment: static-metric and
data-flow-graph preparation, two-stage graph-enhanced fine-tuning, downstream classification, the
ablation study, the significance tests, and a reporting layer that reconstructs the RQ1–RQ4 answers
from the result tables. The code is the source of truth for the methodology.

### Layout

| Part | Contents |
|---|---|
| 0. Requirements | Package list and `requirements.txt` |
| I. Configuration | Every path, switch and **hyper-parameter** in one cell |
| Prerequisites | What is available locally, what will be fetched, and the one credential (TabPFN) |
| II. Data preparation | Ten robust static metrics + token data-flow graphs |
| III. Model & pipeline | Stage-1 fine-tuning, Stage-2 extraction, 10 classifiers, ablation, statistics |
| IV. Results | RQ1–RQ4 narrative, comparison tables, SOTA table (writes itself) |

### Fetches what it needs; runs the experiment

The dataset and the fine-tuned checkpoints live in the public repo and are **fetched from GitHub on
demand** — only what the enabled stages need. By default the notebook performs a **full compute run**:
it fetches the data + checkpoints, extracts the graph-enhanced embeddings, trains the ten downstream
classifiers, runs both ablations with the significance tests, and then the reporting layer prints the
RQ1–RQ4 answers from the freshly computed tables. Metric and data-flow-graph extraction are off because
the hosted data already carries those columns; Stage-1 fine-tuning is off because the released
checkpoints are used. The full run needs a GPU and the packages in Part 0.

Every stage is guarded: on a machine without the stack (or offline), the compute stages skip and the
report still renders from the result tables **embedded in the notebook**, so it never crashes. The
only credential that cannot be fetched is the optional **TabPFN token**.

---
## Part 0 — Requirements

Install the stack below for the default compute run (Parts II–III). If you only want the report, the
tables are embedded in the notebook, so Part IV renders with just `pandas`. Downloads use the standard
library (`urllib`). The install line is commented; the cell also writes `requirements.txt`.

In [ ]:
# Uncomment and run once on a fresh machine (e.g. Colab):
#
# !pip install torch transformers tree_sitter tree_sitter_java \
#              scikit-learn xgboost lightgbm catboost statsmodels \
#              pandas numpy matplotlib seaborn tqdm
# !pip install tabpfn-client        # optional: hosted TabPFN (needs a token)
#
# A full `git clone` of the repo already includes the data and (via Git LFS) the checkpoints, so no
# download is needed. If you only have this notebook, leave AUTO_FETCH_FROM_GITHUB on and it will
# pull whatever a stage needs from GitHub.

REQUIREMENTS = [
    "torch", "transformers", "tree_sitter", "tree_sitter_java",
    "scikit-learn", "xgboost", "lightgbm", "catboost", "statsmodels",
    "pandas", "numpy", "matplotlib", "seaborn", "tqdm",
    "tabpfn-client",   # optional: hosted TabPFN
]
from pathlib import Path as _P
(_P.cwd() / "requirements.txt").write_text("\n".join(REQUIREMENTS) + "\n", encoding="utf-8")
print("wrote requirements.txt")

---
## Part I — Configuration

The single source of truth: paths, run switches, every hyper-parameter (backbone + all ten
classifiers), and the GitHub source used to fetch the data and checkpoints when they are not present
locally. Hyper-parameter values are taken verbatim from the experiment notebooks.

To use your **own** files, set `data_dir` / `checkpoint_dir` (or the individual names) and turn
`AUTO_FETCH_FROM_GITHUB` off. To pin an exact snapshot, set `github_ref` to a commit SHA instead of the
branch name.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional
import urllib.request, urllib.parse
import subprocess, shutil

PROJECT_ROOT = Path.cwd()


@dataclass
class ExperimentConfig:
    # ============================== local paths ==================================================
    project_root:   Path = PROJECT_ROOT
    data_dir:       Optional[Path] = None      # default: <root>/processed data
    checkpoint_dir: Optional[Path] = None      # default: <root>/model checkpoints
    results_dir:    Optional[Path] = None
    output_dir:     Optional[Path] = None
    figure_dir:     Optional[Path] = None

    data_subdir: str = "processed data"
    ckpt_subdir: str = "model checkpoints"
    train_pkl: str = "train_df.pkl"
    val_pkl:   str = "val_df.pkl"
    test_pkl:  str = "test_df.pkl"
    # checkpoint locations relative to checkpoint_dir (same layout as the repo)
    ckpt_rel: Dict = field(default_factory=lambda: {
        "full":      "codet5p-fsp-pytorch-default-v1/codet5p_best.pt",
        "no_gnn":    "codet5p-fsp-pytorch-no-gnn-v1/codet5p_nognn_best.pt",
        "no_extras": "codet5p-fsp-pytorch-no-handcrafted-metrics-v1/codet5p_noextras_best.pt",
    })
    result_files: List[str] = field(default_factory=lambda: [
        "main_classifier_results", "v1_v2_comparison", "ablation_full_metrics", "ablation_summary",
        "statistical_tests", "contribution_analysis", "partial_ablation_rf", "per_class_auc",
        "rf_class_report", "sota_comparison",
    ])

    # ============================== GitHub source (fetch when not local) =========================
    AUTO_FETCH_FROM_GITHUB: bool = True
    FETCH_VIA_CLONE:        bool = True    # git clone the repo (with LFS) once, then use local files
    github_owner: str = "dsslab-projects"
    github_repo:  str = "Graph-Enhanced-CodeT5-Embeddings-for-Software-Fault-Severity-Prediction"
    github_ref:   str = "main"   # branch name or a commit SHA

    # ============================== run switches (all False -> reproduce from results/) ==========
    RUN_METRIC_EXTRACTION:    bool = False   # the hosted data already carries the *_robust columns
    RUN_DFG_EXTRACTION:       bool = False   # the hosted data already carries the data_flow_graph column
    RUN_STAGE1_TRAINING:      bool = False   # use the released checkpoints instead of retraining
    RUN_EMBEDDING_EXTRACTION: bool = True
    RUN_DOWNSTREAM:           bool = True
    RUN_PARTIAL_ABLATION:     bool = True
    RUN_FULL_ABLATION:        bool = True

    # ============================== backbone / model =============================================
    checkpoint:   str   = "Salesforce/codet5p-110m-embedding"
    num_classes:  int   = 4
    hidden_size:  int   = 768
    dropout:      float = 0.26316509043013103
    gnn_layers:   int   = 3
    gnn_heads:    int   = 8
    gnn_last_k:   int   = 4
    trans_last_k: int   = 4

    # ============================== tokenisation / batching ======================================
    max_seq_len: int = 1024
    batch_size:  int = 6
    accum_steps: int = 2
    num_workers: int = 0    # 0 avoids DataLoader worker noise on Kaggle/Colab

    # ============================== Stage-1 fine-tuning ==========================================
    lr:                   float = 8.5e-05
    weight_decay:         float = 0.00010045918919119982
    warmup_ratio:         float = 0.15
    epochs:               int   = 12
    freeze_bottom_blocks: int   = 4
    label_smoothing:      float = 0.1
    grad_clip:            float = 1.0

    # ============================== downstream classifiers (tuned; single source of truth) =======
    knn_params: Dict = field(default_factory=lambda: {"n_neighbors": 3, "weights": "distance", "p": 1})
    svm_params: Dict = field(default_factory=lambda: {"C": 2.5, "kernel": "rbf", "gamma": "scale", "probability": True})
    gnb_params: Dict = field(default_factory=lambda: {"var_smoothing": 1e-8})
    dt_params: Dict = field(default_factory=lambda: {"max_depth": 14, "min_samples_split": 4})
    rf_params: Dict = field(default_factory=lambda: {
        "n_estimators": 1500, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1,
        "max_features": 0.3, "class_weight": "balanced_subsample", "bootstrap": True,
        "oob_score": True, "n_jobs": -1})
    ada_params: Dict = field(default_factory=lambda: {"n_estimators": 500, "learning_rate": 0.85})
    xgb_params: Dict = field(default_factory=lambda: {
        "objective": "multi:softprob", "num_class": 4, "eval_metric": "mlogloss",
        "max_depth": 20, "eta": 0.24627429143007107, "subsample": 0.45321841598276075,
        "colsample_bytree": 0.7227038914198726, "lambda": 0.06640744768945579,
        "alpha": 0.21504472646446163, "tree_method": "hist", "verbosity": 0})
    xgb_rounds:     int = 3000
    xgb_early_stop: int = 30
    lgb_params: Dict = field(default_factory=lambda: {
        "objective": "multiclass", "num_class": 4, "metric": "multi_logloss",
        "learning_rate": 0.085, "max_depth": 7, "verbosity": -1})
    lgb_rounds: int = 1200
    cat_params: Dict = field(default_factory=lambda: {
        "learning_rate": 0.093, "depth": 7, "l2_leaf_reg": 7.07, "iterations": 829})

    tabpfn_use_cloud: bool = True
    tabpfn_token_env: str  = "TABPFN_TOKEN"
    tabpfn_token:     str  = ""   # <-- paste your Prior Labs TabPFN token here, OR leave "" and add it
                                  #     as a Kaggle Secret / env var named TABPFN_TOKEN (see below).
                                  #     Leave it unset to skip TabPFN; the other nine classifiers still run.

    # ============================== features / misc ==============================================
    extra_features: List[str] = field(default_factory=lambda: [
        "sloc_robust", "proxy_indentation_robust", "mcCabe_robust", "mcClure_robust",
        "nested_block_depth_robust", "difficulty_robust", "maintainability_index_robust",
        "fan_out_robust", "readability_robust", "effort_robust"])
    sev_names: List[str] = field(default_factory=lambda: [
        "Severity 0", "Severity 1", "Severity 2", "Severity 3"])
    seed: int = 42

    # ============================== derived ======================================================
    def __post_init__(self):
        self.data_dir       = Path(self.data_dir       or (self.project_root / self.data_subdir))
        self.checkpoint_dir = Path(self.checkpoint_dir or (self.project_root / self.ckpt_subdir))
        self.output_dir     = Path(self.output_dir     or (self.project_root / "outputs"))
        self.results_dir    = Path(self.results_dir    or (self.output_dir / "results"))
        self.figure_dir     = Path(self.figure_dir     or (self.output_dir / "figures"))
        for d in (self.results_dir, self.output_dir, self.figure_dir):
            d.mkdir(parents=True, exist_ok=True)

    @property
    def extra_dim(self) -> int:
        return len(self.extra_features)

    @property
    def final_dim(self) -> int:
        return self.hidden_size + self.extra_dim   # 768 + 10 = 778

    def split_path(self, name: str) -> Path:
        return self.data_dir / {"train": self.train_pkl, "val": self.val_pkl, "test": self.test_pkl}[name]

    def ckpt_path(self, which: str) -> Path:
        return self.checkpoint_dir / self.ckpt_rel[which]

    # ---- GitHub fetch (raw URL; falls back to the LFS media host for checkpoint pointers) --------
    def _fetch(self, repo_rel_path: str, dest: Path) -> Path:
        dest.parent.mkdir(parents=True, exist_ok=True)
        enc = urllib.parse.quote(repo_rel_path)
        base = f"{self.github_owner}/{self.github_repo}/{self.github_ref}/{enc}"
        raw   = f"https://raw.githubusercontent.com/{base}"
        media = f"https://media.githubusercontent.com/media/{base}"
        # Git-LFS files (checkpoints) are served by the media host; raw returns only the pointer text.
        # Non-LFS files (pickles, csv) come straight from raw.
        urls = [media, raw] if repo_rel_path.endswith(".pt") else [raw, media]
        last = None
        for url in urls:
            try:
                print(f"  fetching {repo_rel_path}")
                urllib.request.urlretrieve(url, dest)
                if dest.stat().st_size < 2048 and dest.read_bytes()[:32].startswith(b"version https://git-lfs"):
                    last = "received a Git-LFS pointer, not the file"
                    continue                      # try the other host
                return dest
            except Exception as exc:
                last = exc
        raise RuntimeError(f"could not download {repo_rel_path} ({last}). "
                           f"For LFS checkpoints, check the repo's LFS bandwidth quota.")

    def materialize(self):
        """Clone the repo once (with Git LFS) and place the data + checkpoints into the local dirs.

        This is the reliable path for the LFS checkpoints: `git clone` (with git-lfs) pulls the real
        files, whereas per-file raw URLs only return LFS pointers. Runs at most once and is a no-op if
        the files are already present (e.g. a full clone).
        """
        if getattr(self, "_materialized", False):
            return
        need_data = not all(self.split_path(s).exists() for s in ("train", "val", "test"))
        need_ckpt = not all(self.ckpt_path(w).exists() for w in ("full", "no_gnn", "no_extras"))
        if not (need_data or need_ckpt):
            self._materialized = True
            return
        work = self.project_root / "_gh_repo"
        try:
            if not (work / ".git").exists():
                # Git-LFS is required to pull the real checkpoints; Kaggle images often lack it.
                if subprocess.run(["git", "lfs", "version"], capture_output=True).returncode != 0:
                    print("  installing git-lfs ...")
                    subprocess.run(["apt-get", "-qq", "install", "-y", "git-lfs"], check=False)
                subprocess.run(["git", "lfs", "install"], check=False)
                url = f"https://github.com/{self.github_owner}/{self.github_repo}.git"
                print(f"  cloning {self.github_repo} (branch {self.github_ref}) with Git LFS ...")
                subprocess.run(["git", "clone", "--depth", "1", "--branch", self.github_ref,
                                url, str(work)], check=True)
                subprocess.run(["git", "-C", str(work), "lfs", "pull"], check=False)
            if need_data:
                self.data_dir.mkdir(parents=True, exist_ok=True)
                for f in (work / self.data_subdir).glob("*.pkl"):
                    shutil.copy(f, self.data_dir / f.name)
            if need_ckpt:
                for w in ("full", "no_gnn", "no_extras"):
                    src, dst = work / self.ckpt_subdir / self.ckpt_rel[w], self.ckpt_path(w)
                    if src.exists() and not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True)
                        shutil.move(str(src), str(dst))
            for w in ("full", "no_gnn", "no_extras"):
                pt = self.ckpt_path(w)
                if pt.exists() and pt.stat().st_size < 1_000_000:
                    print(f"  WARNING: {pt.name} is only {pt.stat().st_size} B - looks like an LFS "
                          f"pointer. git-lfs did not pull the real file. Run  !apt-get install -y "
                          f"git-lfs  in a cell, delete the _gh_repo folder, and re-run this cell.")
            shutil.rmtree(work, ignore_errors=True)      # free the clone's .git/LFS store
            self._materialized = True
            print("  data + checkpoints ready (from clone)")
        except Exception as exc:
            print(f"  clone-materialize failed ({exc}); falling back to per-file download")

    def ensure_data(self, name: str) -> Optional[Path]:
        p = self.split_path(name)
        if p.exists():
            return p
        if self.FETCH_VIA_CLONE:
            self.materialize()
            if p.exists():
                return p
        if self.AUTO_FETCH_FROM_GITHUB:
            fn = {"train": self.train_pkl, "val": self.val_pkl, "test": self.test_pkl}[name]
            return self._fetch(f"{self.data_subdir}/{fn}", p)
        return None

    def ensure_checkpoint(self, which: str) -> Optional[Path]:
        p = self.ckpt_path(which)
        if p.exists():
            return p
        if self.FETCH_VIA_CLONE:
            self.materialize()
            if p.exists():
                return p
        if self.AUTO_FETCH_FROM_GITHUB:
            return self._fetch(f"{self.ckpt_subdir}/{self.ckpt_rel[which]}", p)
        return None

cfg = ExperimentConfig()
print(f"Project root : {cfg.project_root}")
print(f"Fused dim    : {cfg.hidden_size} + {cfg.extra_dim} = {cfg.final_dim}")
print(f"Stage-1      : lr={cfg.lr}  wd={cfg.weight_decay:.2e}  warmup={cfg.warmup_ratio}  "
      f"epochs={cfg.epochs}  batch={cfg.batch_size}x{cfg.accum_steps}  freeze={cfg.freeze_bottom_blocks}")
print(f"Downstream   : RF{cfg.rf_params['n_estimators']}/{cfg.rf_params['max_features']}  "
      f"XGB eta={cfg.xgb_params['eta']:.3f}  LGB lr={cfg.lgb_params['learning_rate']}  "
      f"Cat {cfg.cat_params['iterations']}it  KNN k={cfg.knn_params['n_neighbors']}  SVM C={cfg.svm_params['C']}")
print(f"Source       : github:{cfg.github_owner}/{cfg.github_repo}@{cfg.github_ref}  "
      f"(AUTO_FETCH={cfg.AUTO_FETCH_FROM_GITHUB})")
print(f"Switches     : EMB/DOWNSTREAM/PARTIAL/FULL=True -> full compute run; "
      f"metric/DFG off (already in data); Stage-1 off (uses released checkpoints)")

In [ ]:
# --------------------------------------------------------------------------------------------------
# Lightweight imports used everywhere. Part IV needs only these; the heavy stack loads in Part III.
# --------------------------------------------------------------------------------------------------
import warnings, logging, json, math, re, time, pickle
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("severity")

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    _PLOTTING = True
except Exception:
    _PLOTTING = False

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("core imports ready | plotting:", _PLOTTING)

---
## Prerequisites — what is available, what will be fetched

One place to see what the notebook has and what it will pull from GitHub. The dataset and checkpoints
are fetched on demand for the stages that need them (and reused if a full clone already has them). The
result tables are computed by the run; if a stage cannot run, the reporter falls back to the tables
embedded in the notebook. The only credential is the optional TabPFN token.

| Resource | Needed for | Default source |
|---|---|---|
| Dataset splits | any compute stage | local `processed data/`, else fetched from GitHub |
| Checkpoints (`.pt`, LFS) | extraction, downstream, ablation | local `model checkpoints/`, else fetched from GitHub |
| Result CSVs | the Part IV report | local `results/`, else fetched from GitHub |
| PyTorch + Transformers | fine-tuning, extraction | `pip install` (GPU recommended) |
| CodeT5+ backbone | fine-tuning, extraction | Hugging Face download on first use |
| tree-sitter + Java grammar | regenerating data-flow graphs | `pip install tree_sitter tree_sitter_java` |
| **TabPFN token** | the TabPFN classifier only | Config `tabpfn_token`, a Kaggle Secret `TABPFN_TOKEN` (Add-ons -> Secrets), or env `TABPFN_TOKEN` |

In [ ]:
import importlib.util, os

def _have(pkg: str) -> bool:
    try:
        return importlib.util.find_spec(pkg) is not None
    except Exception:
        return False

HAS_TORCH        = _have("torch")
HAS_TRANSFORMERS = _have("transformers")
HAS_TREE_SITTER  = _have("tree_sitter") and _have("tree_sitter_java")
HAS_SKLEARN      = _have("sklearn")
HAS_XGB          = _have("xgboost")
HAS_LGB          = _have("lightgbm")
HAS_CAT          = _have("catboost")
HAS_STATSMODELS  = _have("statsmodels")
HAS_TABPFN       = _have("tabpfn_client") or _have("tabpfn")

# local presence
HAS_DATA        = all(cfg.split_path(s).exists() for s in ("train", "val", "test"))
HAS_CKPT_FULL   = cfg.ckpt_path("full").exists()
HAS_CKPT_NOGNN  = cfg.ckpt_path("no_gnn").exists()
HAS_CKPT_NOEXTR = cfg.ckpt_path("no_extras").exists()

# available-or-fetchable (what the guards actually check)
_AF = cfg.AUTO_FETCH_FROM_GITHUB
DATA_READY        = HAS_DATA or _AF
CKPT_FULL_READY   = HAS_CKPT_FULL or _AF
CKPT_ALL_READY    = (HAS_CKPT_FULL and HAS_CKPT_NOGNN and HAS_CKPT_NOEXTR) or _AF
def resolve_tabpfn_token(cfg):
    """TabPFN token from (1) the Config field, (2) a Kaggle Secret, (3) an environment variable."""
    if cfg.tabpfn_token:
        return cfg.tabpfn_token
    try:                                          # Kaggle: Add-ons -> Secrets -> TABPFN_TOKEN
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret(cfg.tabpfn_token_env)
        if tok:
            return tok
    except Exception:
        pass
    return os.environ.get(cfg.tabpfn_token_env, "")

HAS_TABPFN_TOKEN  = bool(resolve_tabpfn_token(cfg))


def _state(local):
    return "local" if local else ("fetch" if _AF else "MISSING")

def _mark(ok):
    return "[ ok ]" if ok else "[ -- ]"

print("=" * 70)
print("  PREREQUISITE STATUS")
print("=" * 70)
print("  packages")
for label, ok in [("torch", HAS_TORCH), ("transformers", HAS_TRANSFORMERS),
                  ("tree-sitter (+java)", HAS_TREE_SITTER), ("scikit-learn", HAS_SKLEARN),
                  ("xgboost", HAS_XGB), ("lightgbm", HAS_LGB), ("catboost", HAS_CAT),
                  ("statsmodels", HAS_STATSMODELS), ("tabpfn", HAS_TABPFN)]:
    print(f"    {_mark(ok)}  {label}")
print("  data / checkpoints / results  (AUTO_FETCH =", _AF, ")")
print(f"    dataset splits         : {_state(HAS_DATA)}")
print(f"    checkpoint full        : {_state(HAS_CKPT_FULL)}")
print(f"    checkpoint no-GNN      : {_state(HAS_CKPT_NOGNN)}")
print(f"    checkpoint no-metrics  : {_state(HAS_CKPT_NOEXTR)}")
print("  credentials")
print(f"    {_mark(HAS_TABPFN_TOKEN)}  TabPFN token (${cfg.tabpfn_token_env})")
print("=" * 70)
print("  Missing packages / offline machines only disable their own stage; Part IV runs from results/.")


def skip(reason: str):
    print(f"  >> SKIPPED: {reason}")

---
## Part II — Data preparation

Ten robustly-scaled static metrics and a token-level data-flow graph per method, both stored on the
split dataframes. The supplied pickles already contain them, so these stages stay off by default.

### II.1 Static code metrics

Ten lightweight, lexical metrics. The calculator is copied verbatim from the study's export.

In [ ]:
# ============================ static code metric calculator (verbatim) ============================
from collections import Counter
from statistics import pstdev
from typing import Any

class CodeMetricsCalculator:
    def __init__(self):
        pass
    
    def calculate_all_metrics(self, code: str) -> Dict[str, Any]:
        """Calculate all code metrics for a given Java code snippet"""
        metrics = {}
        
        # SLOC
        metrics['sloc'] = self.calculate_sloc(code)
        
        # Proxy Indentation
        metrics['proxy_indentation'] = self.calculate_proxy_indentation(code)
        
        # McCabe Cyclomatic Complexity
        metrics['mcCabe'] = self.calculate_mcCabe(code)
        
        # Nested Block Depth
        metrics['nested_block_depth'] = self.calculate_nbd(code)
        
        # McClure Complexity
        mcclure_metrics = self.calculate_mcClure(code)
        metrics['mcClure'] = mcclure_metrics['MCLC']
        metrics['mcClure_NVAR'] = mcclure_metrics['NVAR']
        metrics['mcClure_NCOMP'] = mcclure_metrics['NCOMP']
        
        # Halstead Metrics
        halstead_metrics = self.calculate_halstead_metrics(code)
        metrics['difficulty'] = halstead_metrics.get('difficulty', 0)
        metrics['effort'] = halstead_metrics.get('effort', 0)
        
        # Maintainability Index
        metrics['maintainability_index'] = self.calculate_maintenance_index(
            code, halstead_metrics, metrics['mcCabe'], metrics['sloc']
        )
        
        # Readability
        metrics['readability'] = self.calculate_readability(code)
        
        # Fan Out
        metrics['fan_out'] = self.calculate_fan_out(code)
        
        return metrics
    
    def calculate_sloc(self, code: str) -> int:
        """Calculate Source Lines of Code"""
        # Remove block comments
        code = re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)
        lines = code.splitlines()
        
        blank = 0
        for line in lines:
            stripped = line.strip()
            if stripped == "" or stripped.startswith("//"):
                blank += 1
        
        return len(lines) - blank
    
    def calculate_proxy_indentation(self, code: str) -> float:
        """Calculate indentation complexity"""
        code_wo_block = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
        indent_values = []
        
        for line in code_wo_block.splitlines():
            stripped = line.strip()
            if stripped == "" or stripped.startswith("//"):
                continue
            
            space_count = 0
            is_blank_line = True
            for ch in line:
                if ch == ' ':
                    space_count += 1
                elif ch == '\t':
                    space_count += 8
                else:
                    is_blank_line = False
                    break
            
            if not is_blank_line:
                indent_values.append(space_count)
        
        if not indent_values:
            return 0.0
        
        sd = pstdev(indent_values)
        return round(sd, 2)
    
    def calculate_mcCabe(self, code: str) -> int:
        """Calculate McCabe cyclomatic complexity"""
        if_stmts = len(re.findall(r'\bif\b', code))
        for_stmts = len(re.findall(r'\bfor\b', code))
        while_stmts = len(re.findall(r'\bwhile\b', code))
        do_stmts = len(re.findall(r'\bdo\b', code))
        catch_clauses = len(re.findall(r'\bcatch\b', code))
        ternary_exprs = len(re.findall(r'\?.*?:', code))
        foreach_stmts = len(re.findall(r'for\s*\(', code))
        switch_entries = len(re.findall(r'\bcase\b', code))
        break_stmts = len(re.findall(r'\bbreak\b', code))
        continue_stmts = len(re.findall(r'\bcontinue\b', code))
        throw_stmts = len(re.findall(r'\bthrow\b', code))
        
        total = (
            if_stmts + for_stmts + foreach_stmts + while_stmts + do_stmts +
            switch_entries + catch_clauses + ternary_exprs +
            break_stmts + continue_stmts + throw_stmts + 1
        )
        return total
    
    def calculate_nbd(self, code: str) -> int:
        """Calculate Nested Block Depth"""
        depth = 0
        max_depth = 0
        tokens = re.findall(r'\b(if|switch|try|for|while|do|synchronized|class|void|public|private)\b|[{]|[}]', code)
        
        for token in tokens:
            if token and token.strip():
                keyword = token.strip()
                if keyword in ["if", "switch", "try", "for", "while", "do", "synchronized", "class", "void", "public", "private"]:
                    depth += 1
                    max_depth = max(max_depth, depth)
                elif keyword == "{":
                    depth += 1
                    max_depth = max(max_depth, depth)
                elif keyword == "}":
                    depth = max(0, depth - 1)
        
        return max_depth - 1 if max_depth > 0 else 0
    
    def calculate_mcClure(self, code: str) -> Dict[str, int]:
        """Calculate McClure complexity metrics"""
        namesNVR = set()
        countCompare = 0
        maxCompareInOneExpression = 0
        
        # Extract conditions
        conditional_exprs = []
        conditional_exprs += re.findall(r'if\s*\((.*?)\)', code, re.DOTALL)
        conditional_exprs += re.findall(r'while\s*\((.*?)\)', code, re.DOTALL)
        conditional_exprs += re.findall(r'do\s*\{.*?\}\s*while\s*\((.*?)\)', code, re.DOTALL)
        conditional_exprs += re.findall(r'for\s*\((.*?);(.*?);(.*?)\)', code, re.DOTALL)
        conditional_exprs += re.findall(r'\?(.*?)\:', code, re.DOTALL)
        conditional_exprs += re.findall(r'switch\s*\((.*?)\)', code, re.DOTALL)
        
        for expr in conditional_exprs:
            if isinstance(expr, tuple):
                expr = " ".join(expr)
            
            # Count comparisons
            sum_compares = 1
            sum_compares += expr.count("&&")
            sum_compares += expr.count("||")
            
            if maxCompareInOneExpression < sum_compares:
                maxCompareInOneExpression = sum_compares
            
            countCompare += sum_compares
            
            # Extract variable names
            tokens = re.findall(r'[A-Za-z_]\w*', expr)
            for token in tokens:
                if not token.isupper():
                    namesNVR.add(token)
        
        # Correction for switch statements
        num_switch_selectors = len(re.findall(r'switch\s*\(', code))
        num_cases = len(re.findall(r'case\s+[^:]+:', code))
        countCompare = (countCompare - num_switch_selectors) + num_cases
        
        NVAR = len(namesNVR)
        NCOMP = countCompare
        MCLC = NVAR + NCOMP
        
        return {
            'NVAR': NVAR,
            'NCOMP': NCOMP,
            'MCLC': MCLC,
            'max_compare': maxCompareInOneExpression
        }
    
    def calculate_halstead_metrics(self, code: str) -> Dict[str, float]:
        """Calculate Halstead metrics"""
        try:
            # Simple approximation since full Halstead requires complex parsing
            operators = Counter()
            operands = Counter()
            
            # Count basic operators and operands
            operators['='] = len(re.findall(r'=', code))
            operators['+'] = len(re.findall(r'\+', code))
            operators['-'] = len(re.findall(r'-', code))
            operators['*'] = len(re.findall(r'\*', code))
            operators['/'] = len(re.findall(r'/', code))
            operators['%'] = len(re.findall(r'%', code))
            operators['=='] = len(re.findall(r'==', code))
            operators['!='] = len(re.findall(r'!=', code))
            operators['<'] = len(re.findall(r'<', code))
            operators['>'] = len(re.findall(r'>', code))
            operators['<='] = len(re.findall(r'<=', code))
            operators['>='] = len(re.findall(r'>=', code))
            operators['&&'] = len(re.findall(r'&&', code))
            operators['||'] = len(re.findall(r'\|\|', code))
            operators['!'] = len(re.findall(r'!', code))
            operators['++'] = len(re.findall(r'\+\+', code))
            operators['--'] = len(re.findall(r'--', code))
            
            # Count variables and literals as operands
            variables = re.findall(r'\b([a-zA-Z_][a-zA-Z0-9_]*)\b', code)
            for var in variables:
                if var not in ['if', 'else', 'for', 'while', 'do', 'switch', 'case', 'default', 'return', 'class', 'void', 'public', 'private', 'protected', 'static', 'final', 'int', 'double', 'float', 'boolean', 'char', 'String']:
                    operands[var] += 1
            
            # Count numeric literals
            numbers = re.findall(r'\b\d+\.?\d*\b', code)
            for num in numbers:
                operands[num] += 1
            
            # Count string literals
            strings = re.findall(r'"[^"]*"', code)
            for s in strings:
                operands[s] += 1
            
            n1 = len(operators)
            n2 = len(operands)
            N1 = sum(operators.values())
            N2 = sum(operands.values())
            
            if n1 + n2 == 0:
                return {'difficulty': 0, 'effort': 0, 'volume': 0}
            
            volume = (N1 + N2) * math.log2(n1 + n2)
            difficulty = (n1 / 2) * (N2 / n2) if n2 > 0 else 0
            effort = difficulty * volume
            
            return {
                'difficulty': round(difficulty, 2),
                'effort': round(effort, 2),
                'volume': round(volume, 2)
            }
            
        except Exception as e:
            print(f"Error calculating Halstead metrics: {e}")
            return {'difficulty': 0, 'effort': 0, 'volume': 0}
    
    def calculate_maintenance_index(self, code: str, halstead_metrics: Dict[str, float], mcCabe: int, sloc: int) -> float:
        """Calculate Maintainability Index"""
        volume = halstead_metrics.get('volume', 0)
        if sloc == 0:
            return 0
        
        try:
            value = 171 - (5.2 * math.log(volume if volume > 0 else 1)) \
                    - (0.23 * mcCabe) \
                    - (16.2 * math.log(sloc if sloc > 0 else 1))
            return round(value, 2)
        except:
            return 0
    
    def calculate_readability(self, code: str) -> float:
        """Calculate readability score"""
        lines = [line.strip() for line in code.splitlines() if line.strip()]
        if not lines:
            return 0.0
        
        avg_length = sum(len(line) for line in lines) / len(lines)
        return round(avg_length, 2)
    
    def calculate_fan_out(self, code: str) -> int:
        """Calculate fan-out (method calls)"""
        method_calls = re.findall(r'\b\w+\s*\(', code)
        keywords = {"if(", "for(", "while(", "switch(", "catch(", "return(", "new("}
        filtered = [m for m in method_calls if m not in keywords]
        return len(filtered)

# Main processing function


### II.2 Robust scaling and fusion

`RobustScaler` (median / IQR) suits the right-skewed metric distributions. `compute_metric_frame`
turns raw code into the ten `*_robust` columns and can reuse a fitted scaler across splits.

In [ ]:
if HAS_SKLEARN:
    from sklearn.preprocessing import RobustScaler

# -- robust scaler used in the study (adds a '<metric>_robust' column per input metric) ------------
def robust_standardize_metrics(df, metrics_columns=None, scaler=None):
    """
    Scale metrics using sklearn's RobustScaler (robust to outliers).
    Adds new columns with suffix '_robust'.
    """
    if metrics_columns is None:
        metrics_columns = df.select_dtypes(include=['number']).columns.tolist()

    if scaler is None:
        scaler = RobustScaler()
        scaled = scaler.fit_transform(df[metrics_columns])
    else:
        scaled = scaler.transform(df[metrics_columns])

    robust_df = df.copy()
    for i, col in enumerate(metrics_columns):
        robust_df[col + "_robust"] = scaled[:, i]

    return robust_df, scaler



RAW_METRIC_COLS = [
    "sloc", "proxy_indentation", "mcCabe", "mcClure", "nested_block_depth",
    "difficulty", "maintainability_index", "fan_out", "readability", "effort",
]


def compute_metric_frame(df, code_column="code", scaler=None):
    calc = CodeMetricsCalculator()
    records = []
    for code in df[code_column].tolist():
        try:
            records.append(calc.calculate_all_metrics(code))
        except Exception as exc:
            logger.warning("metric calc failed: %s", exc)
            records.append({})
    metrics_df = pd.DataFrame(records).reindex(columns=RAW_METRIC_COLS).fillna(0.0)
    out = pd.concat([df.reset_index(drop=True), metrics_df], axis=1)
    out, scaler = robust_standardize_metrics(out, metrics_columns=RAW_METRIC_COLS, scaler=scaler)
    return out, scaler


_raw_csv = cfg.data_dir / "methods.csv"
if not cfg.RUN_METRIC_EXTRACTION:
    print("RUN_METRIC_EXTRACTION=False -> using metrics already stored on the split dataframes")
elif not (HAS_SKLEARN and _raw_csv.exists()):
    skip("need scikit-learn and a raw code csv at data_dir/methods.csv")
else:
    _raw, _ = compute_metric_frame(pd.read_csv(_raw_csv), code_column="code")
    _raw.to_pickle(cfg.data_dir / "methods_with_metrics.pkl")
    print("metrics computed for", len(_raw), "methods")

### II.3 Token data-flow graphs

Each method is parsed with tree-sitter and its def-use relations traced into tuples. The supplied
pickles already carry the `data_flow_graph` column. Helpers and the Java extractor are verbatim.

In [ ]:
# ========================= data-flow-graph extraction helpers (verbatim) ==========================
import tokenize
from io import StringIO

if HAS_TREE_SITTER:
    from tree_sitter import Language, Parser
    import tree_sitter_java
    _JAVA_LANGUAGE = Language(tree_sitter_java.language())


def remove_comments_and_docstrings(source,lang):
    if lang in ['python']:
        """
        Returns 'source' minus comments and docstrings.
        """
        io_obj = StringIO(source)
        out = ""
        prev_toktype = tokenize.INDENT
        last_lineno = -1
        last_col = 0
        for tok in tokenize.generate_tokens(io_obj.readline):
            token_type = tok[0]
            token_string = tok[1]
            start_line, start_col = tok[2]
            end_line, end_col = tok[3]
            ltext = tok[4]
            if start_line > last_lineno:
                last_col = 0
            if start_col > last_col:
                out += (" " * (start_col - last_col))
            # Remove comments:
            if token_type == tokenize.COMMENT:
                pass
            # This series of conditionals removes docstrings:
            elif token_type == tokenize.STRING:
                if prev_toktype != tokenize.INDENT:
            # This is likely a docstring; double-check we're not inside an operator:
                    if prev_toktype != tokenize.NEWLINE:
                        if start_col > 0:
                            out += token_string
            else:
                out += token_string
            prev_toktype = token_type
            last_col = end_col
            last_lineno = end_line
        temp=[]
        for x in out.split('\n'):
            if x.strip()!="":
                temp.append(x)
        return '\n'.join(temp)
    elif lang in ['ruby']:
        return source
    else:
        def replacer(match):
            s = match.group(0)
            if s.startswith('/'):
                return " " # note: a space and not an empty string
            else:
                return s
        pattern = re.compile(
            r'//.*?$|/\*.*?\*/|\'(?:\\.|[^\\\'])*\'|"(?:\\.|[^\\"])*"',
            re.DOTALL | re.MULTILINE
        )
        temp=[]
        for x in re.sub(pattern, replacer, source).split('\n'):
            if x.strip()!="":
                temp.append(x)
        return '\n'.join(temp)


def tree_to_token_index(root_node):
    if (len(root_node.children)==0 or root_node.type=='string') and root_node.type!='comment':
        return [(root_node.start_point,root_node.end_point)]
    else:
        code_tokens=[]
        for child in root_node.children:
            code_tokens+=tree_to_token_index(child)
        return code_tokens


def tree_to_variable_index(root_node,index_to_code):
    if (len(root_node.children)==0 or root_node.type=='string') and root_node.type!='comment':
        index=(root_node.start_point,root_node.end_point)
        _,code=index_to_code[index]
        if root_node.type!=code:
            return [(root_node.start_point,root_node.end_point)]
        else:
            return []
    else:
        code_tokens=[]
        for child in root_node.children:
            code_tokens+=tree_to_variable_index(child,index_to_code)
        return code_tokens    


def index_to_code_token(index,code):
    start_point=index[0]
    end_point=index[1]
    if start_point[0]==end_point[0]:
        s=code[start_point[0]][start_point[1]:end_point[1]]
    else:
        s=""
        s+=code[start_point[0]][start_point[1]:]
        for i in range(start_point[0]+1,end_point[0]):
            s+=code[i]
        s+=code[end_point[0]][:end_point[1]]   
    return s



def DFG_java(root_node,index_to_code,states):
    assignment=['assignment_expression']
    def_statement=['variable_declarator']
    increment_statement=['update_expression']
    if_statement=['if_statement','else']
    for_statement=['for_statement']
    enhanced_for_statement=['enhanced_for_statement']
    while_statement=['while_statement']
    do_first_statement=[]    
    states=states.copy()
    if (len(root_node.children)==0 or root_node.type=='string') and root_node.type!='comment':
        idx,code=index_to_code[(root_node.start_point,root_node.end_point)]
        if root_node.type==code:
            return [],states
        elif code in states:
            return [(code,idx,'comesFrom',[code],states[code].copy())],states
        else:
            if root_node.type=='identifier':
                states[code]=[idx]
            return [(code,idx,'comesFrom',[],[])],states
    elif root_node.type in def_statement:
        name=root_node.child_by_field_name('name')
        value=root_node.child_by_field_name('value')
        DFG=[]
        if value is None:
            indexs=tree_to_variable_index(name,index_to_code)
            for index in indexs:
                idx,code=index_to_code[index]
                DFG.append((code,idx,'comesFrom',[],[]))
                states[code]=[idx]
            return sorted(DFG,key=lambda x:x[1]),states
        else:
            name_indexs=tree_to_variable_index(name,index_to_code)
            value_indexs=tree_to_variable_index(value,index_to_code)
            temp,states=DFG_java(value,index_to_code,states)
            DFG+=temp            
            for index1 in name_indexs:
                idx1,code1=index_to_code[index1]
                for index2 in value_indexs:
                    idx2,code2=index_to_code[index2]
                    DFG.append((code1,idx1,'comesFrom',[code2],[idx2]))
                states[code1]=[idx1]   
            return sorted(DFG,key=lambda x:x[1]),states
    elif root_node.type in assignment:
        left_nodes=root_node.child_by_field_name('left')
        right_nodes=root_node.child_by_field_name('right')
        DFG=[]
        temp,states=DFG_java(right_nodes,index_to_code,states)
        DFG+=temp            
        name_indexs=tree_to_variable_index(left_nodes,index_to_code)
        value_indexs=tree_to_variable_index(right_nodes,index_to_code)        
        for index1 in name_indexs:
            idx1,code1=index_to_code[index1]
            for index2 in value_indexs:
                idx2,code2=index_to_code[index2]
                DFG.append((code1,idx1,'computedFrom',[code2],[idx2]))
            states[code1]=[idx1]   
        return sorted(DFG,key=lambda x:x[1]),states
    elif root_node.type in increment_statement:
        DFG=[]
        indexs=tree_to_variable_index(root_node,index_to_code)
        for index1 in indexs:
            idx1,code1=index_to_code[index1]
            for index2 in indexs:
                idx2,code2=index_to_code[index2]
                DFG.append((code1,idx1,'computedFrom',[code2],[idx2]))
            states[code1]=[idx1]
        return sorted(DFG,key=lambda x:x[1]),states   
    elif root_node.type in if_statement:
        DFG=[]
        current_states=states.copy()
        others_states=[]
        flag=False
        tag=False
        if 'else' in root_node.type:
            tag=True
        for child in root_node.children:
            if 'else' in child.type:
                tag=True
            if child.type not in if_statement and flag is False:
                temp,current_states=DFG_java(child,index_to_code,current_states)
                DFG+=temp
            else:
                flag=True
                temp,new_states=DFG_java(child,index_to_code,states)
                DFG+=temp
                others_states.append(new_states)
        others_states.append(current_states)
        if tag is False:
            others_states.append(states)
        new_states={}
        for dic in others_states:
            for key in dic:
                if key not in new_states:
                    new_states[key]=dic[key].copy()
                else:
                    new_states[key]+=dic[key]
        for key in new_states:
            new_states[key]=sorted(list(set(new_states[key])))
        return sorted(DFG,key=lambda x:x[1]),new_states
    elif root_node.type in for_statement:
        DFG=[]
        for child in root_node.children:
            temp,states=DFG_java(child,index_to_code,states)
            DFG+=temp
        flag=False
        for child in root_node.children:
            if flag:
                temp,states=DFG_java(child,index_to_code,states)
                DFG+=temp                
            elif child.type=="local_variable_declaration":
                flag=True
        dic={}
        for x in DFG:
            if (x[0],x[1],x[2]) not in dic:
                dic[(x[0],x[1],x[2])]=[x[3],x[4]]
            else:
                dic[(x[0],x[1],x[2])][0]=list(set(dic[(x[0],x[1],x[2])][0]+x[3]))
                dic[(x[0],x[1],x[2])][1]=sorted(list(set(dic[(x[0],x[1],x[2])][1]+x[4])))
        DFG=[(x[0],x[1],x[2],y[0],y[1]) for x,y in sorted(dic.items(),key=lambda t:t[0][1])]
        return sorted(DFG,key=lambda x:x[1]),states
    elif root_node.type in enhanced_for_statement:
        name=root_node.child_by_field_name('name')
        value=root_node.child_by_field_name('value')
        body=root_node.child_by_field_name('body')
        DFG=[]
        for i in range(2):
            temp,states=DFG_java(value,index_to_code,states)
            DFG+=temp       
            name_indexs=tree_to_variable_index(name,index_to_code)
            value_indexs=tree_to_variable_index(value,index_to_code)        
            for index1 in name_indexs:
                idx1,code1=index_to_code[index1]
                for index2 in value_indexs:
                    idx2,code2=index_to_code[index2]
                    DFG.append((code1,idx1,'computedFrom',[code2],[idx2]))
                states[code1]=[idx1]   
            temp,states=DFG_java(body,index_to_code,states)
            DFG+=temp                       
        dic={}
        for x in DFG:
            if (x[0],x[1],x[2]) not in dic:
                dic[(x[0],x[1],x[2])]=[x[3],x[4]]
            else:
                dic[(x[0],x[1],x[2])][0]=list(set(dic[(x[0],x[1],x[2])][0]+x[3]))
                dic[(x[0],x[1],x[2])][1]=sorted(list(set(dic[(x[0],x[1],x[2])][1]+x[4])))
        DFG=[(x[0],x[1],x[2],y[0],y[1]) for x,y in sorted(dic.items(),key=lambda t:t[0][1])]
        return sorted(DFG,key=lambda x:x[1]),states
    elif root_node.type in while_statement:  
        DFG=[]
        for i in range(2):
            for child in root_node.children:
                temp,states=DFG_java(child,index_to_code,states)
                DFG+=temp    
        dic={}
        for x in DFG:
            if (x[0],x[1],x[2]) not in dic:
                dic[(x[0],x[1],x[2])]=[x[3],x[4]]
            else:
                dic[(x[0],x[1],x[2])][0]=list(set(dic[(x[0],x[1],x[2])][0]+x[3]))
                dic[(x[0],x[1],x[2])][1]=sorted(list(set(dic[(x[0],x[1],x[2])][1]+x[4])))
        DFG=[(x[0],x[1],x[2],y[0],y[1]) for x,y in sorted(dic.items(),key=lambda t:t[0][1])]
        return sorted(DFG,key=lambda x:x[1]),states        
    else:
        DFG=[]
        for child in root_node.children:
            if child.type in do_first_statement:
                temp,states=DFG_java(child,index_to_code,states)
                DFG+=temp
        for child in root_node.children:
            if child.type not in do_first_statement:
                temp,states=DFG_java(child,index_to_code,states)
                DFG+=temp
        
        return sorted(DFG,key=lambda x:x[1]),states


In [ ]:
def extract_dfg(code: str, lang: str = "java"):
    if not HAS_TREE_SITTER:
        raise RuntimeError("tree-sitter is not installed; cannot extract data-flow graphs")
    try:
        cleaned = remove_comments_and_docstrings(code, lang)
        parser = Parser(_JAVA_LANGUAGE)
        tree = parser.parse(bytes(cleaned, "utf8"))
        tokens_index = tree_to_token_index(tree.root_node)
        code_lines = cleaned.split("\n")
        code_tokens = [index_to_code_token(idx, code_lines) for idx in tokens_index]
        index_to_code = {index: (i, tok) for i, (index, tok) in enumerate(zip(tokens_index, code_tokens))}
        dfg, _ = DFG_java(tree.root_node, index_to_code, {})
        return dfg
    except Exception as exc:
        logger.warning("failed to parse method: %s", exc)
        return []


def add_dfg_to_dataframe(df):
    df = df.copy()
    df["data_flow_graph"] = df["code"].apply(lambda c: extract_dfg(c, "java"))
    return df


_metrics_pkl = cfg.data_dir / "methods_with_metrics.pkl"
if not cfg.RUN_DFG_EXTRACTION:
    print("RUN_DFG_EXTRACTION=False -> reusing the stored `data_flow_graph` column")
elif not (HAS_TREE_SITTER and _metrics_pkl.exists()):
    skip("need tree-sitter and data_dir/methods_with_metrics.pkl")
else:
    _src = add_dfg_to_dataframe(pd.read_pickle(_metrics_pkl))
    _src.to_pickle(cfg.data_dir / "processed_data_with_dfgs.pkl")
    print("data-flow graphs computed for", len(_src), "methods")

---
## Part III — Model and the two-stage pipeline

Stage 1 fine-tunes CodeT5+; Stage 2 extracts a 768-d embedding, appends the ten metrics, and trains
the downstream classifiers. Every stage is guarded and fetches the data / checkpoint it needs.

In [ ]:
# --------------------------------------------------------------------------------------------------
# Deep-learning stack. If torch is unavailable, shims let the definitions load so Part IV still runs.
# --------------------------------------------------------------------------------------------------
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
    from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
    try:
        from tqdm.auto import tqdm
    except Exception:
        def tqdm(x, *a, **k):
            return x
    DL_AVAILABLE = True
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("PyTorch ready | device:", DEVICE)
except Exception:
    DL_AVAILABLE, DEVICE = False, "cpu"

    class _TorchShim:
        @staticmethod
        def no_grad():
            def deco(fn):
                return fn
            return deco

        def __getattr__(self, name):
            def _stub(*a, **k):
                raise RuntimeError("PyTorch not installed")
            return _stub

    class _NNShim:
        Module = object

        def __getattr__(self, name):
            def _stub(*a, **k):
                raise RuntimeError("PyTorch not installed")
            return _stub

    torch, nn, F, Dataset = _TorchShim(), _NNShim(), _NNShim(), object
    def tqdm(x, *a, **k):
        return x
    print("PyTorch unavailable -> Part III compute disabled; Part IV still runs.")

In [ ]:
def build_graph_tensors(dfg, seq_len):
    edges, edge_types = [], []
    if seq_len > 1:
        edges.extend([[i, i + 1] for i in range(seq_len - 1)])
        edge_types.extend([6] * (seq_len - 1))
    if isinstance(dfg, list):
        for node in dfg:
            if len(node) == 5:
                _, tgt, _, _, srcs = node
                for s in srcs:
                    s, t = int(s), int(tgt)
                    if s < seq_len and t < seq_len:
                        edges.append([s, t]); edge_types.append(0)
    if edges:
        return (torch.tensor(edges, dtype=torch.long).t(), torch.tensor(edge_types, dtype=torch.long))
    return (torch.zeros((2, 0), dtype=torch.long), torch.zeros(0, dtype=torch.long))


class BugSeverityDataset(Dataset):
    def __init__(self, df, tokenizer, cfg):
        self.df, self.tok, self.cfg = df.reset_index(drop=True), tokenizer, cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tok(row["code"], truncation=True, max_length=self.cfg.max_seq_len, return_tensors="pt")
        ids, mask = enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0)
        ei, et = build_graph_tensors(row["data_flow_graph"], ids.size(0))
        return {"input_ids": ids, "attention_mask": mask, "edge_index": ei, "edge_types": et,
                "extras": torch.tensor(row[self.cfg.extra_features].values.astype(np.float32)),
                "label": torch.tensor(int(row["label"]), dtype=torch.long)}


def collate_fn(batch):
    max_len, B = max(b["input_ids"].size(0) for b in batch), len(batch)
    ids_pad = torch.zeros(B, max_len, dtype=torch.long)
    mask_pad = torch.zeros(B, max_len, dtype=torch.long)
    for i, b in enumerate(batch):
        l = b["input_ids"].size(0)
        ids_pad[i, :l], mask_pad[i, :l] = b["input_ids"], b["attention_mask"]
    edges, types, offset = [], [], 0
    for b in batch:
        if b["edge_index"].size(1) > 0:
            edges.append(b["edge_index"] + offset); types.append(b["edge_types"])
        offset += max_len
    edge_index = torch.cat(edges, dim=1) if edges else torch.zeros((2, 0), dtype=torch.long)
    edge_types = torch.cat(types) if types else torch.zeros(0, dtype=torch.long)
    return {"input_ids": ids_pad, "attention_mask": mask_pad, "edge_index": edge_index,
            "edge_types": edge_types, "extras": torch.stack([b["extras"] for b in batch]),
            "label": torch.stack([b["label"] for b in batch])}


class DataModule:
    """Loads the three splits (fetching them from GitHub if needed) and builds DataLoaders."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.checkpoint, trust_remote_code=True)
        paths = {s: cfg.ensure_data(s) for s in ("train", "val", "test")}
        missing = [s for s, p in paths.items() if p is None]
        if missing:
            raise FileNotFoundError(f"missing splits {missing} (AUTO_FETCH_FROM_GITHUB is off)")
        self.train_df = pd.read_pickle(paths["train"])
        self.val_df = pd.read_pickle(paths["val"])
        self.test_df = pd.read_pickle(paths["test"])
        logger.info("splits | train=%d val=%d test=%d",
                    len(self.train_df), len(self.val_df), len(self.test_df))
        self.train_loader = self._loader(self.train_df, shuffle=True)
        self.val_loader = self._loader(self.val_df)
        self.test_loader = self._loader(self.test_df)

    def _loader(self, df, shuffle=False):
        ds = BugSeverityDataset(df, self.tokenizer, self.cfg)
        return DataLoader(ds, batch_size=self.cfg.batch_size, shuffle=shuffle,
                          collate_fn=collate_fn, num_workers=self.cfg.num_workers, pin_memory=True)

### III.1 Graph attention and the graph-enhanced encoder

Multi-head attention over the token graph (edge-type embeddings, residual + layer norm, weights
normalised across the batched edge set); the encoder averages the last four hidden states before
graph propagation and mean-pools into a 768-d embedding.

In [ ]:
class MultiHeadGraphAttention(nn.Module):
    def __init__(self, in_features, out_features, num_heads=4, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = out_features // num_heads
        self.edge_type_embed = nn.Embedding(10, out_features)
        self.W_q = nn.Linear(in_features, out_features)
        self.W_k = nn.Linear(in_features, out_features)
        self.W_v = nn.Linear(in_features, out_features)
        self.W_e = nn.Linear(out_features, out_features)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(out_features)

    def forward(self, node_features, edge_index, edge_types):
        N, E = node_features.shape[0], edge_index.shape[1]
        Q = self.W_q(node_features).reshape(N, self.num_heads, self.head_dim)
        K = self.W_k(node_features).reshape(N, self.num_heads, self.head_dim)
        V = self.W_v(node_features).reshape(N, self.num_heads, self.head_dim)
        edge_emb = self.W_e(self.edge_type_embed(edge_types)).reshape(E, self.num_heads, self.head_dim)
        src, dst = edge_index[0], edge_index[1]
        attn_scores = (Q[dst] * (K[src] + edge_emb)).sum(dim=-1) / np.sqrt(self.head_dim)
        attn_weights = self.dropout(torch.softmax(attn_scores, dim=0))
        out = torch.zeros(N, self.num_heads, self.head_dim, device=node_features.device)
        out.index_add_(0, dst, attn_weights.unsqueeze(-1) * V[src])
        return self.layer_norm(node_features + out.reshape(N, -1))


class CodeGraphNeuralNetwork(nn.Module):
    def __init__(self, hidden_size, num_layers=3, num_heads=8, dropout=0.1, last_k=4):
        super().__init__()
        self.last_k = last_k
        self.layers = nn.ModuleList([
            MultiHeadGraphAttention(hidden_size, hidden_size, num_heads, dropout)
            for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, node_features, edge_index, edge_types):
        x, layer_outputs = node_features, []
        for layer in self.layers:
            x = self.dropout(layer(x, edge_index, edge_types))
            layer_outputs.append(x)
        k = min(self.last_k, len(layer_outputs))
        return torch.stack(layer_outputs[-k:], dim=0).mean(dim=0)

In [ ]:
class GraphEnhancedCodeT5p(nn.Module):
    """CodeT5+ backbone + graph attention + metric fusion with an MLP severity head."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        base_cfg = AutoConfig.from_pretrained(cfg.checkpoint, trust_remote_code=True)
        if not hasattr(base_cfg, "is_decoder"):
            base_cfg.is_decoder = False
        if not hasattr(base_cfg, "is_encoder_decoder"):
            base_cfg.is_encoder_decoder = False
        self.base_model = AutoModel.from_pretrained(cfg.checkpoint, config=base_cfg, trust_remote_code=True)
        self.gnn = CodeGraphNeuralNetwork(cfg.hidden_size, cfg.gnn_layers, cfg.gnn_heads,
                                          cfg.dropout, cfg.gnn_last_k)
        self.fusion = nn.Sequential(nn.Linear(cfg.hidden_size, 256), nn.ReLU(), nn.Dropout(cfg.dropout))
        self.classifier = nn.Sequential(
            nn.Dropout(cfg.dropout), nn.Linear(256 + cfg.extra_dim, 128),
            nn.ReLU(), nn.Dropout(cfg.dropout), nn.Linear(128, cfg.num_classes))

    def _encode(self, input_ids, attention_mask, average_last_k):
        def _run(ids, mask):
            if hasattr(self.base_model, "transformer"):
                out = self.base_model.transformer(input_ids=ids, attention_mask=mask,
                                                  output_hidden_states=average_last_k)
            elif hasattr(self.base_model, "encoder"):
                out = self.base_model.encoder(input_ids=ids, attention_mask=mask,
                                              output_hidden_states=average_last_k)
            else:
                out = self.base_model(input_ids=ids, attention_mask=mask, output_hidden_states=True)
            if average_last_k:
                k = min(self.cfg.trans_last_k, len(out.hidden_states))
                return torch.stack(out.hidden_states[-k:], dim=0).mean(0)
            return out.last_hidden_state

        B, seq_len = input_ids.shape
        if seq_len <= 512:
            return _run(input_ids, attention_mask)
        pad_len = (512 - seq_len % 512) % 512
        if pad_len:
            pid = getattr(self.base_model.config, "pad_token_id", None) or 0
            input_ids = F.pad(input_ids, (0, pad_len), value=pid)
            attention_mask = F.pad(attention_mask, (0, pad_len), value=0)
        _, padded = input_ids.shape
        nc = padded // 512
        out = _run(input_ids.reshape(B * nc, 512), attention_mask.reshape(B * nc, 512))
        return out.reshape(B, padded, -1)[:, :seq_len, :]

    def encode_tokens(self, input_ids, attention_mask):
        return self._encode(input_ids, attention_mask, average_last_k=False)

    def avg_last_k_tokens(self, input_ids, attention_mask):
        return self._encode(input_ids, attention_mask, average_last_k=True)

    def graph_mean_pool(self, input_ids, attention_mask, edge_index, edge_types,
                        use_avg_last_k=True, use_gnn=True):
        B, L = input_ids.shape
        tok = self.avg_last_k_tokens(input_ids, attention_mask) if use_avg_last_k \
            else self.encode_tokens(input_ids, attention_mask)
        flat = tok.reshape(B * L, -1)
        if use_gnn and edge_index.size(1) > 0:
            flat = self.gnn(flat, edge_index, edge_types)
        graph_feat = flat.reshape(B, L, -1)
        mask_exp = attention_mask.unsqueeze(-1).float()
        return (graph_feat * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask, edge_index, edge_types, extras):
        emb = self.graph_mean_pool(input_ids, attention_mask, edge_index, edge_types)
        return self.classifier(torch.cat([self.fusion(emb), extras], dim=-1))

### III.2 Stage 1 — fine-tuning

Class-balanced cross-entropy with 0.1 label smoothing, AdamW, linear warmup, gradient clipping and
two-step accumulation, bottom four blocks frozen. All settings come from the Configuration cell. Runs
only when `RUN_STAGE1_TRAINING` is on and PyTorch + the dataset are available.

In [ ]:
if HAS_SKLEARN:
    from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
    from sklearn.metrics import f1_score


class FineTuner:
    def __init__(self, cfg, model, data_module):
        self.cfg, self.model, self.dm = cfg, model.to(DEVICE), data_module
        cw = compute_class_weight("balanced", classes=np.array([0, 1, 2, 3]),
                                  y=data_module.train_df["label"].values)
        self.criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(cw, dtype=torch.float).to(DEVICE), label_smoothing=cfg.label_smoothing)
        for name, param in self.model.base_model.named_parameters():
            if "block" in name:
                nums = [s for s in name.split(".") if s.isdigit()]
                if nums and int(nums[0]) < cfg.freeze_bottom_blocks:
                    param.requires_grad = False
        self.optimizer = AdamW(self.model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        total_steps = (len(self.dm.train_loader) // cfg.accum_steps) * cfg.epochs
        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer, int(total_steps * cfg.warmup_ratio), total_steps)

    def _batch(self, batch):
        t = {k: batch[k].to(DEVICE, non_blocking=True)
             for k in ("input_ids", "attention_mask", "edge_index", "edge_types", "extras", "label")}
        logits = self.model(t["input_ids"], t["attention_mask"], t["edge_index"], t["edge_types"], t["extras"])
        return self.criterion(logits, t["label"]), logits, t["label"]

    def train_epoch(self, epoch):
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        self.optimizer.zero_grad()
        loader = self.dm.train_loader
        for step, batch in enumerate(tqdm(loader, desc=f"epoch {epoch:02d}", leave=False)):
            loss, logits, labels = self._batch(batch)
            (loss / self.cfg.accum_steps).backward()
            if (step + 1) % self.cfg.accum_steps == 0 or (step + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                self.optimizer.step(); self.scheduler.step(); self.optimizer.zero_grad()
            total_loss += loss.item()
            correct += (logits.argmax(1) == labels).sum().item(); total += labels.size(0)
        return total_loss / len(loader), correct / total

    @torch.no_grad()
    def eval_epoch(self, loader):
        self.model.eval()
        preds, trues = [], []
        for batch in tqdm(loader, desc="eval", leave=False):
            _, logits, labels = self._batch(batch)
            preds.extend(logits.argmax(1).cpu().numpy()); trues.extend(labels.cpu().numpy())
        return f1_score(trues, preds, average="macro")

    def fit(self):
        history, best_f1 = [], 0.0
        self.cfg.ckpt_path("full").parent.mkdir(parents=True, exist_ok=True)
        for epoch in range(1, self.cfg.epochs + 1):
            tr_loss, tr_acc = self.train_epoch(epoch)
            val_f1 = self.eval_epoch(self.dm.val_loader)
            history.append({"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc, "val_f1": val_f1})
            flag = ""
            if val_f1 > best_f1:
                best_f1 = val_f1
                torch.save(self.model.state_dict(), self.cfg.ckpt_path("full"))
                flag = "  <- best"
            logger.info("epoch %02d | loss %.4f acc %.4f | val macroF1 %.4f%s",
                        epoch, tr_loss, tr_acc, val_f1, flag)
        pd.DataFrame(history).to_csv(self.cfg.output_dir / "finetuning_history.csv", index=False)
        return best_f1


class ModelLoader:
    """Instantiate GraphEnhancedCodeT5p and load a checkpoint (fetched + cached; loaded once)."""

    _CACHE = {}

    @staticmethod
    def load(cfg, which: str = "full"):
        if which in ModelLoader._CACHE:
            return ModelLoader._CACHE[which]
        pt = cfg.ensure_checkpoint(which)
        if pt is None:
            raise FileNotFoundError(f"checkpoint '{which}' not available (AUTO_FETCH_FROM_GITHUB is off)")
        model = GraphEnhancedCodeT5p(cfg).to(DEVICE)
        try:                                       # PyTorch >= 2.6 defaults weights_only=True
            state = torch.load(pt, map_location=DEVICE, weights_only=False)
        except TypeError:                          # older torch without the weights_only kwarg
            state = torch.load(pt, map_location=DEVICE)
        # Keep only tensors whose shape matches this model. The no-GNN / no-metrics ablation
        # checkpoints have different classification heads; extraction only needs the (matching)
        # backbone + GNN, so drop mismatched head layers instead of letting load_state_dict raise.
        msd = model.state_dict()
        compatible = {k: v for k, v in state.items()
                      if k in msd and tuple(v.shape) == tuple(msd[k].shape)}
        model.load_state_dict(compatible, strict=False)
        model.eval()
        logger.info("loaded %d/%d tensors from '%s' -> %s", len(compatible), len(state), pt.name, DEVICE)
        ModelLoader._CACHE[which] = model
        return model


if not cfg.RUN_STAGE1_TRAINING:
    print("Stage-1 training skipped (RUN_STAGE1_TRAINING=False).")
elif not (DL_AVAILABLE and HAS_TRANSFORMERS and HAS_SKLEARN and DATA_READY):
    skip("need PyTorch + Transformers + scikit-learn + dataset")
else:
    try:
        _dm = DataModule(cfg)
        print("best val Macro F1:", FineTuner(cfg, GraphEnhancedCodeT5p(cfg), _dm).fit())
    except Exception as e:
        skip(f"could not run Stage-1 training: {e}")

### III.3 Stage 2 — extraction, evaluation and the downstream suite

Stage 2 forms the 778-d features; the evaluator computes the full imbalanced metric suite; the suite
reads every hyper-parameter from cfg, fits on train only, and writes the RQ1 / RQ4 tables. Fetches the
dataset and the full checkpoint if they are not local.

In [ ]:
if HAS_SKLEARN:
    from sklearn.preprocessing import label_binarize
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                                 recall_score, f1_score, matthews_corrcoef, cohen_kappa_score,
                                 roc_auc_score, confusion_matrix, classification_report)


class MetricsEvaluator:
    def __init__(self, cfg):
        self.cfg = cfg

    @staticmethod
    def _safe(v, d=4):
        return round(v, d) if not (isinstance(v, float) and np.isnan(v)) else np.nan

    def compute(self, name, y_true, y_pred, y_prob):
        nc = self.cfg.num_classes
        y_bin = label_binarize(y_true, classes=np.arange(nc))
        cm = confusion_matrix(y_true, y_pred)
        sens = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
        gmean = float(np.prod(sens[sens > 0]) ** (1.0 / max(np.sum(sens > 0), 1)))
        try:
            auc_ovr = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
            per_auc = roc_auc_score(y_bin, y_prob, average=None)
        except Exception:
            auc_ovr, per_auc = np.nan, [np.nan] * nc
        row = {"Model": name,
               "Accuracy": self._safe(accuracy_score(y_true, y_pred)),
               "Balanced_Accuracy": self._safe(balanced_accuracy_score(y_true, y_pred)),
               "Precision_macro": self._safe(precision_score(y_true, y_pred, average="macro", zero_division=0)),
               "Recall_macro": self._safe(recall_score(y_true, y_pred, average="macro", zero_division=0)),
               "F1_macro": self._safe(f1_score(y_true, y_pred, average="macro", zero_division=0)),
               "Precision_weighted": self._safe(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
               "Recall_weighted": self._safe(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
               "F1_weighted": self._safe(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
               "MCC": self._safe(matthews_corrcoef(y_true, y_pred)),
               "Cohen_Kappa": self._safe(cohen_kappa_score(y_true, y_pred)),
               "ROC_AUC_OVR_macro": self._safe(auc_ovr),
               "G_Mean": self._safe(gmean)}
        for i, v in enumerate(per_auc):
            row[f"AUC_Sev{i}"] = self._safe(float(v))
        return row


class EmbeddingExtractor:
    def __init__(self, model, cfg, use_gnn=True, use_avg_last_k=True):
        self.model, self.cfg, self.use_gnn, self.use_avg_last_k = model, cfg, use_gnn, use_avg_last_k

    @torch.no_grad()
    def _extract(self, loader):
        embs, labels = [], []
        for batch in tqdm(loader, desc="extract", leave=False):
            ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            ei = batch["edge_index"].to(DEVICE, non_blocking=True)
            et = batch["edge_types"].to(DEVICE, non_blocking=True)
            emb = self.model.graph_mean_pool(ids, mask, ei, et,
                                             use_avg_last_k=self.use_avg_last_k, use_gnn=self.use_gnn)
            embs.append(emb.cpu().numpy()); labels.append(batch["label"].numpy())
        return np.vstack(embs), np.concatenate(labels)

    def extract_all(self, dm, append_extras=True):
        ef, out = self.cfg.extra_features, {}
        for split, loader, df in [("train", dm.train_loader, dm.train_df),
                                  ("val", dm.val_loader, dm.val_df),
                                  ("test", dm.test_loader, dm.test_df)]:
            emb, y = self._extract(loader)
            X = np.hstack([emb, df[ef].values.astype(np.float32)]) if append_extras else emb
            out[f"X_{split}"], out[f"y_{split}"], out[f"emb_{split}"] = X, y, emb
        return out

In [ ]:
def build_tabpfn(cfg):
    """Hosted TabPFN client (token from config / Kaggle Secret / env), else a local model."""
    import sys
    token = resolve_tabpfn_token(cfg)
    if cfg.tabpfn_use_cloud and token:
        if not _have("tabpfn_client"):           # install the client on demand (needs internet)
            print("  installing tabpfn-client ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn-client"], check=False)
        from tabpfn_client import TabPFNClassifier as _Cloud
        import tabpfn_client as _c
        _c.set_access_token(token)
        return _Cloud()
    if _have("tabpfn"):
        from tabpfn import TabPFNClassifier as _Local
        return _Local(ignore_pretraining_limits=True)
    raise RuntimeError("no TabPFN token found (set it in Config / Kaggle Secret / env) and local tabpfn is not installed")


class MLModelSuite:
    def __init__(self, cfg, evaluator):
        self.cfg, self.evaluator, self.results, self.preds = cfg, evaluator, [], {}

    def _add(self, name, y_test, y_pred, y_prob):
        self.results.append(self.evaluator.compute(name, y_test, y_pred, y_prob))
        self.preds[name] = {"y_pred": y_pred, "y_prob": y_prob}
        logger.info("  %-14s F1=%.4f", name, self.results[-1]["F1_macro"])
        if getattr(self, "_bar", None) is not None:
            self._bar.set_postfix_str(f"{name} F1={self.results[-1]['F1_macro']:.3f}"); self._bar.update(1)

    def train_all(self, emb):
        cfg = self.cfg
        X_tr, y_tr = emb["X_train"], emb["y_train"]
        X_va, y_va = emb["X_val"], emb["y_val"]
        X_te, y_te = emb["X_test"], emb["y_test"]
        gpu = DL_AVAILABLE and torch.cuda.is_available()
        self._bar = tqdm(total=10, desc="downstream classifiers")

        def sk(name, clf):
            clf.fit(X_tr, y_tr); self._add(name, y_te, clf.predict(X_te), clf.predict_proba(X_te))

        sk("KNN", KNeighborsClassifier(**cfg.knn_params))
        sk("SVM", SVC(**cfg.svm_params, random_state=cfg.seed))
        sk("Naive Bayes", GaussianNB(**cfg.gnb_params))
        sk("Decision Tree", DecisionTreeClassifier(**cfg.dt_params, random_state=cfg.seed))
        sk("Random Forest", RandomForestClassifier(**cfg.rf_params, random_state=cfg.seed))
        sk("AdaBoost", AdaBoostClassifier(**cfg.ada_params, random_state=cfg.seed))

        if HAS_XGB:
            import xgboost as xgb
            sw = compute_sample_weight("balanced", y=y_tr)
            b = xgb.train(cfg.xgb_params, xgb.DMatrix(X_tr, label=y_tr, weight=sw),
                          num_boost_round=cfg.xgb_rounds,
                          evals=[(xgb.DMatrix(X_va, label=y_va), "val")],
                          early_stopping_rounds=cfg.xgb_early_stop, verbose_eval=False)
            p = b.predict(xgb.DMatrix(X_te)).reshape(-1, cfg.num_classes)
            self._add("XGBoost", y_te, p.argmax(1), p)
        else:
            skip("XGBoost not installed")
        if HAS_LGB:
            import lightgbm as lgb
            b = lgb.train({**cfg.lgb_params, "device": "gpu" if gpu else "cpu"},
                          lgb.Dataset(X_tr, label=y_tr), num_boost_round=cfg.lgb_rounds)
            p = b.predict(X_te); self._add("LightGBM", y_te, p.argmax(1), p)
        else:
            skip("LightGBM not installed")
        if HAS_CAT:
            import catboost as cb
            cw = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr).tolist()
            m = cb.CatBoostClassifier(**cfg.cat_params, task_type="GPU" if gpu else "CPU",
                                      verbose=False, class_weights=cw)
            m.fit(X_tr, y_tr); self._add("CatBoost", y_te, m.predict(X_te).flatten(), m.predict_proba(X_te))
        else:
            skip("CatBoost not installed")
        try:
            tab = build_tabpfn(cfg); tab.fit(X_tr, y_tr)
            self._add("TabPFN", y_te, tab.predict(X_te), tab.predict_proba(X_te))
        except Exception as e:
            skip(f"TabPFN unavailable ({e}); set ${cfg.tabpfn_token_env} or install local tabpfn")
        self._bar.close()
        return pd.DataFrame(self.results).set_index("Model")

    def save_paper_tables(self, df, y_test):
        res = self.cfg.results_dir
        core = ["Accuracy", "Balanced_Accuracy", "Precision_macro", "Recall_macro", "F1_macro",
                "Precision_weighted", "Recall_weighted", "F1_weighted", "MCC", "Cohen_Kappa",
                "ROC_AUC_OVR_macro", "G_Mean"]
        df.reset_index()[["Model"] + core].to_csv(res / "main_classifier_results.csv", index=False)
        auc_cols = [c for c in df.columns if c.startswith("AUC_Sev")]
        auc = df[auc_cols + ["ROC_AUC_OVR_macro"]].reset_index()
        auc.columns = ["Model", "AUC_Sev0", "AUC_Sev1", "AUC_Sev2", "AUC_Sev3", "Macro_AUC"]
        auc.sort_values("Macro_AUC", ascending=False).to_csv(res / "per_class_auc.csv", index=False)
        best = df["F1_macro"].idxmax()
        rep = classification_report(y_test, self.preds[best]["y_pred"],
                                    target_names=[f"Severity {i}" for i in range(self.cfg.num_classes)],
                                    output_dict=True, zero_division=0)
        rows = [[f"Severity {i}", rep[f"Severity {i}"]["precision"], rep[f"Severity {i}"]["recall"],
                 rep[f"Severity {i}"]["f1-score"], int(rep[f"Severity {i}"]["support"])]
                for i in range(self.cfg.num_classes)]
        rows += [["Macro average", rep["macro avg"]["precision"], rep["macro avg"]["recall"],
                  rep["macro avg"]["f1-score"], int(rep["macro avg"]["support"])],
                 ["Weighted average", rep["weighted avg"]["precision"], rep["weighted avg"]["recall"],
                  rep["weighted avg"]["f1-score"], int(rep["weighted avg"]["support"])]]
        pd.DataFrame(rows, columns=["Severity_class", "Precision", "Recall", "F1", "Support"]).to_csv(
            res / "rf_class_report.csv", index=False)
        logger.info("wrote RQ1/RQ4 tables to %s", res)


if HAS_SKLEARN:
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.naive_bayes import GaussianNB
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

# ---- shared data + full-model embeddings: built once, reused by the stages below ----------------
_DM = None
_FULL_EMB = None


def get_dm(cfg):
    global _DM
    if _DM is None:
        _DM = DataModule(cfg)          # fetches the splits from GitHub on first use
    return _DM


def get_full_embeddings(cfg):
    global _FULL_EMB
    if _FULL_EMB is None:
        _FULL_EMB = EmbeddingExtractor(ModelLoader.load(cfg, "full"), cfg).extract_all(get_dm(cfg))
    return _FULL_EMB


# ---- Stage-2 embedding extraction (RUN_EMBEDDING_EXTRACTION) --------------------------------------
if not cfg.RUN_EMBEDDING_EXTRACTION:
    print("Embedding extraction skipped (RUN_EMBEDDING_EXTRACTION=False).")
elif not (DL_AVAILABLE and HAS_TRANSFORMERS and DATA_READY and CKPT_FULL_READY):
    skip("need PyTorch + Transformers + dataset + full checkpoint")
else:
    try:
        _e = get_full_embeddings(cfg)
        np.save(cfg.output_dir / "X_test_778.npy", _e["X_test"])
        print("full-model features:", _e["X_train"].shape, _e["X_val"].shape, _e["X_test"].shape)
    except Exception as e:
        skip(f"could not extract embeddings: {e}")


# ---- Downstream classifier suite (RUN_DOWNSTREAM) -> RQ1 / RQ4 ------------------------------------
if not cfg.RUN_DOWNSTREAM:
    print("Downstream suite skipped (RUN_DOWNSTREAM=False).")
elif not (DL_AVAILABLE and HAS_TRANSFORMERS and HAS_SKLEARN and DATA_READY and CKPT_FULL_READY):
    skip("need PyTorch + Transformers + scikit-learn + dataset + full checkpoint")
else:
    try:
        _emb = get_full_embeddings(cfg)
        _suite = MLModelSuite(cfg, MetricsEvaluator(cfg))
        _res = _suite.train_all(_emb); _suite.save_paper_tables(_res, _emb["y_test"])
        print(_res[["F1_macro", "Accuracy", "MCC"]].sort_values("F1_macro", ascending=False))
    except Exception as e:
        skip(f"could not run downstream suite: {e}")

### III.4 Ablation — partial Stage-2 extraction (RQ3)

Re-extract from the full model under four strategies (graph on/off x average-last-4 vs last-hidden),
fit Random Forest (settings from cfg), write the RQ3 table.

In [ ]:
PARTIAL_STRATEGIES = {
    "GNN + avg-last-4":     dict(use_gnn=True,  use_avg_last_k=True),
    "No GNN + avg-last-4":  dict(use_gnn=False, use_avg_last_k=True),
    "GNN + last hidden":    dict(use_gnn=True,  use_avg_last_k=False),
    "No GNN + last hidden": dict(use_gnn=False, use_avg_last_k=False),
}


def run_partial_ablation(model, dm, cfg):
    evaluator, rows = MetricsEvaluator(cfg), []
    for label, flags in tqdm(PARTIAL_STRATEGIES.items(), total=len(PARTIAL_STRATEGIES),
                             desc="Stage-2 extraction strategies"):
        emb = EmbeddingExtractor(model, cfg, **flags).extract_all(dm)
        rf = RandomForestClassifier(**cfg.rf_params, random_state=cfg.seed)
        rf.fit(emb["X_train"], emb["y_train"])
        m = evaluator.compute(label, emb["y_test"], rf.predict(emb["X_test"]), rf.predict_proba(emb["X_test"]))
        rows.append({"Stage2_extraction_strategy": label, "Accuracy": m["Accuracy"],
                     "F1_macro": m["F1_macro"], "MCC": m["MCC"], "G_Mean": m["G_Mean"]})
    out = pd.DataFrame(rows); out.to_csv(cfg.results_dir / "partial_ablation_rf.csv", index=False)
    return out


if not cfg.RUN_PARTIAL_ABLATION:
    print("Partial ablation skipped (RUN_PARTIAL_ABLATION=False). RQ3 reads cached results/.")
elif not (DL_AVAILABLE and HAS_TRANSFORMERS and HAS_SKLEARN and DATA_READY and CKPT_FULL_READY):
    skip("need PyTorch + Transformers + scikit-learn + dataset + full checkpoint")
else:
    try:
        print(run_partial_ablation(ModelLoader.load(cfg, "full"), get_dm(cfg), cfg))
    except Exception as e:
        skip(f"could not run partial ablation: {e}")

### III.5 Ablation — full component removal, significance, contribution (RQ2)

Four conditions from the three checkpoints (fetched if needed) by toggling graph propagation and
metric fusion; McNemar + permutation tests; marginal component contribution.

In [ ]:
if HAS_STATSMODELS:
    from statsmodels.stats.contingency_tables import mcnemar

FULL_ABLATION_CONDITIONS = {
    "Full (GNN + Metrics)":    dict(ckpt="full",      use_gnn=True,  extras=True),
    "No Static Metrics (GNN)": dict(ckpt="no_extras", use_gnn=True,  extras=False),
    "No GNN (Metrics Only)":   dict(ckpt="no_gnn",    use_gnn=False, extras=True),
    "Pure Transformer":        dict(ckpt="no_gnn",    use_gnn=False, extras=False),
}


def _ablation_suite(emb, cfg, evaluator, condition):
    X_tr, y_tr = emb["X_train"], emb["y_train"]
    X_va, y_va = emb["X_val"], emb["y_val"]
    X_te, y_te = emb["X_test"], emb["y_test"]
    gpu = DL_AVAILABLE and torch.cuda.is_available()
    rows, preds = [], {}

    def rec(name, y_pred, y_prob):
        m = evaluator.compute(name, y_te, y_pred, y_prob)
        m["Condition"], m["Classifier"] = condition, name
        rows.append(m); preds[name] = {"y_pred": y_pred, "y_prob": y_prob}

    rf = RandomForestClassifier(**cfg.rf_params, random_state=cfg.seed)
    rf.fit(X_tr, y_tr); rec("Random Forest", rf.predict(X_te), rf.predict_proba(X_te))
    if HAS_XGB:
        import xgboost as xgb
        sw = compute_sample_weight("balanced", y=y_tr)
        b = xgb.train(cfg.xgb_params, xgb.DMatrix(X_tr, label=y_tr, weight=sw),
                      num_boost_round=cfg.xgb_rounds, evals=[(xgb.DMatrix(X_va, label=y_va), "val")],
                      early_stopping_rounds=cfg.xgb_early_stop, verbose_eval=False)
        p = b.predict(xgb.DMatrix(X_te)).reshape(-1, cfg.num_classes); rec("XGBoost", p.argmax(1), p)
    if HAS_LGB:
        import lightgbm as lgb
        b = lgb.train({**cfg.lgb_params, "device": "gpu" if gpu else "cpu"},
                      lgb.Dataset(X_tr, label=y_tr), num_boost_round=cfg.lgb_rounds)
        p = b.predict(X_te); rec("LightGBM", p.argmax(1), p)
    if HAS_CAT:
        import catboost as cb
        cw = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr).tolist()
        m = cb.CatBoostClassifier(**cfg.cat_params, task_type="GPU" if gpu else "CPU",
                                  verbose=False, class_weights=cw)
        m.fit(X_tr, y_tr); rec("CatBoost", m.predict(X_te).flatten(), m.predict_proba(X_te))
    svm = SVC(**cfg.svm_params, random_state=cfg.seed)
    svm.fit(X_tr, y_tr); rec("SVM", svm.predict(X_te), svm.predict_proba(X_te))
    return rows, preds


class StatisticalAnalyzer:
    def mcnemar(self, y_true, pa, pb):
        ca, cb = (pa == y_true), (pb == y_true)
        table = [[0, int(np.sum(ca & ~cb))], [int(np.sum(~ca & cb)), 0]]
        r = mcnemar(table, exact=False, correction=True)
        return r.statistic, r.pvalue

    def permutation(self, y_true, pa, pb, n_perm=500):
        obs = (f1_score(y_true, pa, average="macro", zero_division=0)
               - f1_score(y_true, pb, average="macro", zero_division=0))
        combined, n, count = np.concatenate([pa, pb]), len(pa), 0
        for _ in range(n_perm):
            np.random.shuffle(combined)
            d = (f1_score(y_true, combined[:n], average="macro", zero_division=0)
                 - f1_score(y_true, combined[n:], average="macro", zero_division=0))
            if abs(d) >= abs(obs):
                count += 1
        return count / n_perm

    def all_pairs(self, all_preds, y_test, classifier="Random Forest"):
        rows, conds = [], list(all_preds.keys())
        for i in range(len(conds)):
            for j in range(i + 1, len(conds)):
                a, b = conds[i], conds[j]
                if classifier not in all_preds[a] or classifier not in all_preds[b]:
                    continue
                pa, pb = all_preds[a][classifier]["y_pred"], all_preds[b][classifier]["y_pred"]
                chi2, p = self.mcnemar(y_test, pa, pb)
                fa = f1_score(y_test, pa, average="macro", zero_division=0)
                fb = f1_score(y_test, pb, average="macro", zero_division=0)
                rows.append({"Condition_A": a, "Condition_B": b, "Classifier": classifier,
                             "F1_A": round(fa, 4), "F1_B": round(fb, 4), "Delta_F1": round(fa - fb, 4),
                             "McNemar_chi2": round(chi2, 4), "McNemar_p": round(p, 4),
                             "Permutation_p": round(self.permutation(y_test, pa, pb), 4),
                             "Significant": "Y" if p < 0.05 else "N"})
        return pd.DataFrame(rows)


class ContributionAnalyzer:
    KEY_METRICS = ["F1_macro", "MCC", "ROC_AUC_OVR_macro", "Balanced_Accuracy"]
    COND = {"A": "Full (GNN + Metrics)", "B": "No Static Metrics (GNN)",
            "C": "No GNN (Metrics Only)", "D": "Pure Transformer"}

    def compute(self, df_full):
        rows = []
        for clf in df_full["Classifier"].unique():
            sub = df_full[df_full["Classifier"] == clf].set_index("Condition")
            for m in self.KEY_METRICS:
                if m not in sub.columns:
                    continue
                v = lambda k: sub.loc[self.COND[k], m] if self.COND[k] in sub.index else np.nan
                fA, fB, fC, fD = v("A"), v("B"), v("C"), v("D")
                rows.append({"Classifier": clf, "Metric": m,
                             "Full_A": round(float(fA), 4), "No_Metrics_B": round(float(fB), 4),
                             "No_GNN_C": round(float(fC), 4), "Pure_D": round(float(fD), 4),
                             "Delta_GNN_contribution": round(float(fA - fC), 4),
                             "Delta_Metrics_contribution": round(float(fA - fB), 4),
                             "Delta_GNN_standalone": round(float(fB - fD), 4),
                             "Delta_Metrics_standalone": round(float(fC - fD), 4),
                             "Delta_vs_Pure": round(float(fA - fD), 4),
                             "Synergy": round(float(fA - fD - (fB - fD) - (fC - fD)), 4)})
        return pd.DataFrame(rows)


def run_full_ablation(dm, cfg):
    evaluator, all_rows, all_preds, y_test = MetricsEvaluator(cfg), [], {}, None
    for label, spec in tqdm(FULL_ABLATION_CONDITIONS.items(), total=len(FULL_ABLATION_CONDITIONS),
                            desc="ablation conditions"):
        model = ModelLoader.load(cfg, spec["ckpt"])
        emb = EmbeddingExtractor(model, cfg, use_gnn=spec["use_gnn"], use_avg_last_k=True) \
            .extract_all(dm, append_extras=spec["extras"])
        rows, preds = _ablation_suite(emb, cfg, evaluator, label)
        all_rows.extend(rows); all_preds[label] = preds; y_test = emb["y_test"]
    df_full = pd.DataFrame(all_rows)
    df_full.to_csv(cfg.results_dir / "ablation_full_metrics.csv", index=False)
    (df_full.sort_values("F1_macro", ascending=False).groupby("Condition").first().reset_index()
     .to_csv(cfg.results_dir / "ablation_summary.csv", index=False))
    best = df_full.groupby("Classifier")["F1_macro"].mean().idxmax()
    if HAS_STATSMODELS:
        StatisticalAnalyzer().all_pairs(all_preds, y_test, best).to_csv(
            cfg.results_dir / "statistical_tests.csv", index=False)
    ContributionAnalyzer().compute(df_full).to_csv(cfg.results_dir / "contribution_analysis.csv", index=False)
    return df_full


if not cfg.RUN_FULL_ABLATION:
    print("Full ablation skipped (RUN_FULL_ABLATION=False). RQ2 reads cached results/.")
elif not (DL_AVAILABLE and HAS_TRANSFORMERS and HAS_SKLEARN and DATA_READY and CKPT_ALL_READY):
    skip("need PyTorch + Transformers + scikit-learn + dataset + all three checkpoints")
else:
    try:
        print(run_full_ablation(get_dm(cfg), cfg).groupby("Condition")["F1_macro"].max())
    except Exception as e:
        skip(f"could not run full ablation: {e}")

---
## Part IV — Results (the report writes itself)

Each cell reads the freshly computed tables (or, if a stage did not run, the copies embedded in the
notebook) and derives the manuscript's findings from the numbers. The final cell writes
`outputs/paper_observations.md`. This part needs only pandas, so the report always renders.

In [ ]:
# ------------------------------------------------------------------------------------------------
# Reporting layer - reads ONLY the tables this run computed (outputs/results). Nothing about this
# experiment is hard-coded: if a stage did not run, its RQ says so. The only fixed numbers are the
# PUBLISHED prior-work values below (earlier CodeBERT study + SOTA reference), cited for comparison
# and clearly labelled as such - they are not produced by this notebook.
# ------------------------------------------------------------------------------------------------
PRIOR_WORK_EARLIER = {   # earlier CodeBERT+metrics study, fixed split -> (Macro F1, accuracy). Cited.
    "Random Forest": (0.7285, 0.7928), "KNN": (0.7331, 0.7968), "XGBoost": (0.7481, 0.8048),
    "SVM": (0.7275, 0.7908), "LightGBM": (0.7257, 0.7968), "CatBoost": (0.7298, 0.7869),
    "Naive Bayes": (0.7029, 0.7530), "AdaBoost": (0.6353, 0.6514), "TabPFN": (0.7208, 0.7829),
}
PRIOR_WORK_SOTA = [      # (Study, Representation, Macro_F1, Weighted_F1, ROC_AUC, MCC). Cited.
    ("Method-level SOTA reference", "Metric-only Random Forest", "-", "0.54", "0.65", "0.21"),
    ("Method-level SOTA reference", "CodeBERT classifier", "-", "0.70", "0.87", "0.51"),
    ("Method-level SOTA reference", "CodeBERT + metrics, ConcatCLS", "-", "0.74", "0.87", "0.56"),
    ("Earlier version", "CodeBERT + metrics, XGBoost", "0.7481", "0.7965", "0.8533 macro", "0.6338"),
]


class ObservationCollector:
    """Turns this run's computed tables into the RQ1-RQ4 findings. Fresh results only."""

    def __init__(self, cfg):
        self.cfg, self.res, self._cache = cfg, cfg.results_dir, {}

    def load(self, name):
        if name not in self._cache:
            p = self.res / f"{name}.csv"
            self._cache[name] = pd.read_csv(p) if p.exists() else None
        return self._cache[name]

    @staticmethod
    def _md_table(df):
        try:
            return df.to_markdown(index=False)
        except Exception:
            return "```\n" + df.to_string(index=False) + "\n```"

    @staticmethod
    def _p(section, lines):
        print("\n" + "=" * 90 + f"\n{section}\n" + "=" * 90)
        for ln in lines:
            print(ln)

    # tables that are produced by a compute run (excludes the prior-work reference tables)
    computed_tables = ["main_classifier_results", "ablation_summary", "statistical_tests",
                       "contribution_analysis", "partial_ablation_rf", "per_class_auc",
                       "rf_class_report"]

    def provenance(self):
        fresh = sum((self.res / f"{n}.csv").exists() for n in self.computed_tables)
        total = len(self.computed_tables)
        if fresh == total:
            return "SOURCE: all tables freshly computed by this run (outputs/results)."
        if fresh == 0:
            return ("SOURCE: no computed tables found - the compute stages did not run. Enable a GPU and "
                    "the Part 0 packages, then Run All. (Nothing is shown from a cache.)")
        return (f"SOURCE: {fresh}/{total} tables freshly computed; the others did not run - their RQ "
                f"prints 'not computed' (check the SKIPPED messages above).")

    def _earlier_comparison(self, main):
        rows = []
        for _, r in main.iterrows():
            base = PRIOR_WORK_EARLIER.get(r["Model"])
            if base is None:
                continue
            ef, ea = base
            rows.append({"Model": r["Model"], "Earlier_F1_Macro": ef, "New_F1_Macro": round(r["F1_macro"], 4),
                         "Delta_F1": round(r["F1_macro"] - ef, 4), "Earlier_Accuracy": ea,
                         "New_Accuracy": round(r["Accuracy"], 4), "Delta_Accuracy": round(r["Accuracy"] - ea, 4)})
        return pd.DataFrame(rows)

    def rq1(self, show=True):
        main = self.load("main_classifier_results")
        if main is None:
            msg = "RQ1 not computed - run RUN_DOWNSTREAM on a GPU."
            if show:
                self._p("RQ1  Does graph-enhanced CodeT5+ improve over the CodeBERT baseline?", [msg])
            return [msg]
        best = main.loc[main["F1_macro"].idxmax()]
        comp = self._earlier_comparison(main)
        lines = [f"Best model: {best['Model']} - Macro F1 {best['F1_macro']:.4f}, accuracy "
                 f"{best['Accuracy']:.4f}, MCC {best['MCC']:.4f}, macro ROC-AUC {best['ROC_AUC_OVR_macro']:.4f}."]
        if len(comp):
            all_up = bool((comp["Delta_F1"] > 0).all())
            top = comp.loc[comp["Delta_F1"].idxmax()]
            eb = comp.loc[comp["Earlier_F1_Macro"].idxmax()]
            nb = comp.loc[comp["New_F1_Macro"].idxmax()]
            lines += [f"vs the earlier CodeBERT baseline (cited): "
                      f"{'every shared classifier improves' if all_up else 'not all improve'} in Macro F1; "
                      f"mean gain {comp['Delta_F1'].mean():+.4f}, largest {top['Model']} ({top['Delta_F1']:+.4f}).",
                      f"New best {nb['Model']} {nb['New_F1_Macro']:.4f} vs earlier best {eb['Model']} "
                      f"{eb['Earlier_F1_Macro']:.4f} = {nb['New_F1_Macro'] - eb['Earlier_F1_Macro']:+.4f}."]
        if show:
            self._p("RQ1  Does graph-enhanced CodeT5+ improve over the CodeBERT baseline?", lines)
            print("\nDownstream classifiers on the full 778-d representation (this run):\n")
            print(main.to_string(index=False))
            if len(comp):
                print("\nComparison with the earlier CodeBERT baseline (earlier = cited prior work):\n")
                print(comp.to_string(index=False))
        return lines

    def rq2(self, show=True):
        summ, stat, contrib = (self.load("ablation_summary"), self.load("statistical_tests"),
                               self.load("contribution_analysis"))
        if summ is None:
            msg = "RQ2 not computed - run RUN_FULL_ABLATION on a GPU."
            if show:
                self._p("RQ2  Which components matter under full component removal?", [msg])
            return [msg]
        by = summ.set_index("Condition")["F1_macro"]
        full, no_met = by.get("Full (GNN + Metrics)", np.nan), by.get("No Static Metrics (GNN)", np.nan)
        no_gnn, pure = by.get("No GNN (Metrics Only)", np.nan), by.get("Pure Transformer", np.nan)
        gnn_loss, met_loss = full - no_gnn, full - no_met
        lines = [f"Full model best Macro F1 {full:.4f}. Removing metrics -> {no_met:.4f} (loss "
                 f"{met_loss:.4f}); removing the GNN -> {no_gnn:.4f} (loss {gnn_loss:.4f}); "
                 f"pure transformer {pure:.4f}.",
                 f"The larger loss is from removing the {'GNN' if gnn_loss > met_loss else 'static metrics'}, "
                 f"so the {'GNN is the dominant new component' if gnn_loss > met_loss else 'metrics dominate'}."]
        if stat is not None:
            sig = stat[stat["McNemar_p"] < 0.05]; ns = stat[stat["McNemar_p"] >= 0.05]
            pair = lambda d: "; ".join(f"{r.Condition_A} vs {r.Condition_B} (p={r.McNemar_p})"
                                       for _, r in d.iterrows())
            lines.append(f"McNemar significant (p<0.05): {pair(sig) if len(sig) else 'none'}.")
            lines.append(f"Not significant: {pair(ns) if len(ns) else 'none'}.")
        if contrib is not None:
            rf = contrib[(contrib["Classifier"] == "Random Forest") & (contrib["Metric"] == "F1_macro")]
            if len(rf):
                r = rf.iloc[0]
                lines.append(f"Contribution (Random Forest, F1): GNN {r['Delta_GNN_contribution']:+.4f} vs "
                             f"metrics {r['Delta_Metrics_contribution']:+.4f}.")
        if show:
            self._p("RQ2  Which components matter under full component removal?", lines)
            print("\nBest classifier per condition:\n"); print(summ.to_string(index=False))
            if stat is not None:
                print("\nStatistical tests:\n"); print(stat.to_string(index=False))
        return lines

    def rq3(self, show=True):
        part = self.load("partial_ablation_rf")
        if part is None:
            msg = "RQ3 not computed - run RUN_PARTIAL_ABLATION on a GPU."
            if show:
                self._p("RQ3  What changes when components are removed only at Stage-2 extraction?", [msg])
            return [msg]
        p = part.set_index("Stage2_extraction_strategy")["F1_macro"]
        full = p.get("GNN + avg-last-4", np.nan)
        lines = [f"Removing the GNN only at extraction lowers Macro F1 from {full:.4f} by just "
                 f"{full - p.get('No GNN + avg-last-4', np.nan):.4f} - far smaller than the full-removal "
                 f"loss, so the graph shapes the learned representation during training.",
                 f"Replacing average-last-4 with the last hidden state costs "
                 f"{full - p.get('GNN + last hidden', np.nan):.4f} Macro F1 even with the GNN on."]
        if show:
            self._p("RQ3  What changes when components are removed only at Stage-2 extraction?", lines)
            print("\nStage-2-only extraction ablation (Random Forest):\n"); print(part.to_string(index=False))
        return lines

    def rq4(self, show=True):
        auc, rep = self.load("per_class_auc"), self.load("rf_class_report")
        if auc is None:
            msg = "RQ4 not computed - run RUN_DOWNSTREAM on a GPU."
            if show:
                self._p("RQ4  How does performance vary across severity classes?", [msg])
            return [msg]
        means = {c: auc[c].mean() for c in ["AUC_Sev0", "AUC_Sev1", "AUC_Sev2", "AUC_Sev3"]}
        easiest, hardest = max(means, key=means.get), min(means, key=means.get)
        lines = [f"Severity separability by mean one-vs-rest AUC: easiest {easiest} ({means[easiest]:.3f}), "
                 f"hardest {hardest} ({means[hardest]:.3f}); the extreme classes stay hardest."]
        if rep is not None:
            r = rep.set_index("Severity_class")
            for c in ["Severity 0", "Severity 3"]:
                if c in r.index:
                    lines.append(f"{c}: precision {r.loc[c, 'Precision']:.2f}, recall {r.loc[c, 'Recall']:.2f}.")
        if show:
            self._p("RQ4  How does performance vary across severity classes?", lines)
            print("\nPer-class one-vs-rest ROC-AUC:\n"); print(auc.to_string(index=False))
            if rep is not None:
                print("\nRandom Forest class-wise report:\n"); print(rep.to_string(index=False))
        return lines

    def sota(self, show=True):
        main = self.load("main_classifier_results")
        rows = list(PRIOR_WORK_SOTA)
        if main is not None:
            b = main.loc[main["F1_macro"].idxmax()]
            rows.append(("This paper", f"CodeT5+ + CGNN + metrics, {b['Model']}",
                         f"{b['F1_macro']:.4f}", f"{b['F1_weighted']:.4f}",
                         f"{b['ROC_AUC_OVR_macro']:.4f} macro", f"{b['MCC']:.4f}"))
        s = pd.DataFrame(rows, columns=["Study", "Representation_best_model", "Macro_F1",
                                        "Weighted_F1", "ROC_AUC", "MCC"])
        if show:
            self._p("Comparison across the research line (prior rows cited; last row is this run)",
                    ["Weighted F1 and MCC rise along the line."])
            print(); print(s.to_string(index=False))
        return s


class Visualizer:
    """Charts built from the freshly computed tables. No-op if a table or matplotlib is missing."""

    COND_SHORT = {"Full (GNN + Metrics)": "Full", "No Static Metrics (GNN)": "No metrics",
                  "No GNN (Metrics Only)": "No GNN", "Pure Transformer": "Pure transf."}

    def __init__(self, report, cfg):
        self.r, self.cfg = report, cfg

    def _done(self, fig, name):
        try:
            fig.savefig(self.cfg.figure_dir / f"{name}.png", dpi=140, bbox_inches="tight")
        except Exception:
            pass
        plt.show(); plt.close(fig)

    def rq1(self):
        if not _PLOTTING:
            return
        main = self.r.load("main_classifier_results")
        if main is None:
            return
        try:
            d = main.sort_values("F1_macro")
            fig, ax = plt.subplots(figsize=(7, 4.2))
            bars = ax.barh(d["Model"], d["F1_macro"], color="#4C78A8")
            ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
            ax.set_xlim(0.6, max(0.86, float(d["F1_macro"].max()) + 0.03))
            ax.set_xlabel("Macro F1"); ax.set_title("RQ1  Macro F1 by classifier (this run)")
            self._done(fig, "rq1_macro_f1")
            comp = self.r._earlier_comparison(main)
            if len(comp):
                comp = comp.sort_values("New_F1_Macro")
                x = np.arange(len(comp)); w = 0.4
                fig, ax = plt.subplots(figsize=(8, 4.2))
                ax.bar(x - w / 2, comp["Earlier_F1_Macro"], w, label="Earlier CodeBERT (cited)", color="#B0B0B0")
                ax.bar(x + w / 2, comp["New_F1_Macro"], w, label="This run (CodeT5+/CGNN)", color="#4C78A8")
                ax.set_xticks(x); ax.set_xticklabels(comp["Model"], rotation=30, ha="right", fontsize=8)
                ax.set_ylim(0.6, 0.87); ax.set_ylabel("Macro F1"); ax.legend(fontsize=8)
                ax.set_title("RQ1  Improvement over the earlier CodeBERT baseline")
                self._done(fig, "rq1_improvement")
        except Exception as e:
            logger.info("rq1 figure skipped: %s", e)

    def rq2(self):
        if not _PLOTTING:
            return
        summ = self.r.load("ablation_summary")
        if summ is None:
            return
        try:
            order = ["Full (GNN + Metrics)", "No Static Metrics (GNN)", "No GNN (Metrics Only)", "Pure Transformer"]
            d = summ.set_index("Condition").reindex([c for c in order if c in list(summ["Condition"])]).reset_index()
            colors = ["#4C78A8", "#F58518", "#54A24B", "#9C27B0"][:len(d)]
            fig, ax = plt.subplots(figsize=(7, 4.2))
            bars = ax.bar(range(len(d)), d["F1_macro"], color=colors)
            ax.bar_label(bars, fmt="%.3f", fontsize=8)
            ax.set_xticks(range(len(d)))
            ax.set_xticklabels([self.COND_SHORT.get(c, c) for c in d["Condition"]], fontsize=9)
            ax.set_ylabel("Macro F1"); ax.set_ylim(0.7, float(d["F1_macro"].max()) + 0.02)
            ax.set_title("RQ2  Best Macro F1 by ablation condition")
            self._done(fig, "rq2_ablation")
        except Exception as e:
            logger.info("rq2 figure skipped: %s", e)

    def rq4(self):
        if not _PLOTTING:
            return
        auc = self.r.load("per_class_auc")
        if auc is None:
            return
        try:
            cols = ["AUC_Sev0", "AUC_Sev1", "AUC_Sev2", "AUC_Sev3"]
            M = auc.set_index("Model")[cols]
            fig, ax = plt.subplots(figsize=(6, max(3, 0.42 * len(M) + 1)))
            im = ax.imshow(M.values, cmap="YlOrRd", vmin=0.5, vmax=1.0, aspect="auto")
            ax.set_xticks(range(4)); ax.set_xticklabels(["Sev0", "Sev1", "Sev2", "Sev3"])
            ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index, fontsize=8)
            for i in range(len(M)):
                for j in range(4):
                    ax.text(j, i, f"{M.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
            fig.colorbar(im, ax=ax, shrink=0.8, label="ROC-AUC")
            ax.set_title("RQ4  Per-class one-vs-rest ROC-AUC")
            self._done(fig, "rq4_per_class_auc")
        except Exception as e:
            logger.info("rq4 figure skipped: %s", e)


report = ObservationCollector(cfg)
viz = Visualizer(report, cfg)
print("ObservationCollector ready | reading tables from", cfg.results_dir)
print(report.provenance())

### RQ1 — Does the graph-enhanced representation improve over the earlier CodeBERT baseline?

In [ ]:
_ = report.rq1()
viz.rq1()

### RQ2 — Which components matter under full component removal?

In [ ]:
_ = report.rq2()
viz.rq2()

### RQ3 — What changes when components are removed only during Stage-2 extraction?

In [ ]:
_ = report.rq3()

### RQ4 — How does performance vary across severity classes?

In [ ]:
_ = report.rq4()
viz.rq4()

### Comparison with the SOTA reference and the earlier version

In [ ]:
_ = report.sota()

### Write the findings to a file

In [ ]:
def write_paper_observations(report, cfg):
    md = ["# Auto-generated results summary", "",
          "_Generated by the reproduction notebook from this run's computed tables._", "",
          f"_{report.provenance()}_", ""]
    main = report.load("main_classifier_results")
    blocks = [
        ("RQ1 - Improvement over the earlier CodeBERT baseline", report.rq1(show=False),
         [("Downstream classifiers (this run)", main),
          ("Earlier-baseline comparison (earlier = cited prior work)",
           report._earlier_comparison(main) if main is not None else None)]),
        ("RQ2 - Full component-removal ablation", report.rq2(show=False),
         [("Best classifier per condition", report.load("ablation_summary")),
          ("Statistical tests", report.load("statistical_tests")),
          ("Component contribution", report.load("contribution_analysis"))]),
        ("RQ3 - Partial Stage-2 extraction ablation", report.rq3(show=False),
         [("Stage-2-only ablation (RF)", report.load("partial_ablation_rf"))]),
        ("RQ4 - Per-class behaviour", report.rq4(show=False),
         [("Per-class ROC-AUC", report.load("per_class_auc")),
          ("RF class report", report.load("rf_class_report"))]),
        ("Comparison across the research line (prior rows cited; last row is this run)",
         ["See table."], [("Research-line comparison", report.sota(show=False))]),
    ]
    for title, lines, tables in blocks:
        md += [f"## {title}", ""] + [f"- {ln}" for ln in lines] + [""]
        for label, df in tables:
            if df is not None and len(df):
                md += [f"**{label}**", "", report._md_table(df), ""]
    out = cfg.output_dir / "paper_observations.md"
    out.write_text("\n".join(md), encoding="utf-8")
    print("wrote", out)
    return out


_ = write_paper_observations(report, cfg)